# Física BR · Taller complementario: Corriente Alterna y Factor de Potencia
### Física BR — Física de Electricidad y Magnetismo
**Profesor:** Juan David Betancur Ríos · **Semestre:** 2026-2

---
*Física interactiva para ingeniería.* Taller complementario del capítulo 8 (Inducción): de la red de 60,0 Hz
a la factura de una planta. Valor eficaz, reactancia, impedancia, potencias activa, reactiva y aparente, y
la corrección del factor de potencia con condensadores. Requiere **Inducción I**. Cada punto parte de un
problema real e incluye esquema, botón **"Mostrar solución"**, cuestionario y exploración con sliders.

**Convenciones:** tres cifras significativas · coma decimal · valores eficaces (rms) salvo que se diga «pico».

**Ecuaciones clave:**
- Valor eficaz:  V_rms = V₀/√2 ,  I_rms = I₀/√2      ·      ω = 2π·f  (red colombiana: f = 60,0 Hz)
- Reactancias:  X_L = ω·L ,  X_C = 1/(ω·C)      ·      Impedancia R–L:  Z = √(R² + X_L²) ,  I_rms = V_rms/Z
- Desfase:  tan φ = X/R      ·      Factor de potencia:  FP = cos φ = P/S
- Potencias:  P = V·I·cos φ (W) ,  Q = V·I·sen φ (VAR) ,  S = V·I (VA) ,  S² = P² + Q²
- Condensador de corrección:  Q_C = V²·ω·C

> **Uso:** `Entorno de ejecución → Ejecutar todo`. Competencias: [CL, UFT, UCM, PRC, ULC].

---
> © 2026 **Juan David Betancur Ríos** (docente de física para ingeniería) y **Nora Yamile Rojas Cataño** (aporte pedagógico y metodología de evaluación). Material educativo de uso académico (Física BR). Todos los derechos reservados.
> *Elaborado con apoyo de herramientas de inteligencia artificial, bajo la supervisión y criterio pedagógico de los autores.*


---
## Punto 1 — Ducha eléctrica: valor eficaz y potencia media (Ing. Eléctrica/Instalaciones)  ★★  [CL, UCM]
Una ducha eléctrica está conectada a un circuito de **120 V (rms)** y **60,0 Hz**, la tensión entre fase y neutro de
las redes de baja tensión colombianas. Su placa dice **P = 3,00 kW** (dato didáctico). La resistencia del calentador
se comporta como una carga puramente resistiva: voltaje y corriente están en fase.

**Valor eficaz:**  V₀ = √2·V_rms ,  I_rms = P/V_rms ,  R = V_rms/I_rms ,  I₀ = √2·I_rms

**Potencia instantánea:**  p(t) = V₀·I₀·sen²(ωt) — oscila entre 0 y V₀·I₀ ;  su promedio es P = V_rms·I_rms

> 🔧 **Aplicación en ingeniería:** los 120 V que marca un multímetro en un tomacorriente son un valor eficaz:
> el voltaje real oscila entre +170 V y −170 V sesenta veces por segundo. Los conductores, los interruptores y
> el cálculo de la factura se dimensionan con valores eficaces, porque son los que dan la potencia media real.
> Confundir pico con eficaz lleva a sobredimensionar o, peor, a subdimensionar los aislamientos.

> 🤖 **TIP-IA — Traductor de representaciones:**
> Pega a un asistente: *"Explícame con la gráfica de p(t) = V₀·I₀·sen²(ωt) por qué la potencia media de una resistencia es la mitad de la potencia pico, y por qué eso hace que V_rms = V₀/√2. Hazme dibujarla primero."* Primero intenta explicarlo tú.


In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as _tk_br
if not getattr(_tk_br.ScalarFormatter, '_coma_br', False):
    # coma decimal en las marcas de los ejes y en el texto de desplazamiento (offset)
    _fmt_br_call, _fmt_br_off = _tk_br.ScalarFormatter.__call__, _tk_br.ScalarFormatter.get_offset
    _tk_br.ScalarFormatter.__call__ = lambda self, x, pos=None: _fmt_br_call(self, x, pos).replace('.', ',')
    def _off_br(self):
        import re as _re_br
        sup = str.maketrans('−-+0123456789', '⁻⁻⁺⁰¹²³⁴⁵⁶⁷⁸⁹')
        t = _fmt_br_off(self).replace('.', ',')
        t = _re_br.sub(r'^1e([−+-]?\d+)', lambda m: '×10' + m.group(1).translate(sup), t)
        return _re_br.sub(r'(\d)e([−+-]?\d+)', lambda m: m.group(1) + '×10' + m.group(2).translate(sup), t)
    _tk_br.ScalarFormatter.get_offset = _off_br
    _tk_br.ScalarFormatter._coma_br = True
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
# --- pantalla angosta (celular) ---
try:
    from IPython.display import HTML as _HTML_br
    display(_HTML_br('<style>'
        '.widget-radio{width:100% !important;max-width:100%;height:auto !important}'
        '.widget-radio-box{width:100%;max-width:100%}'
        '.widget-radio-box label{display:flex !important;flex-direction:row-reverse;justify-content:flex-end;align-items:flex-start;gap:8px;'
        'white-space:normal !important;height:auto !important;line-height:1.35 !important;margin:3px 0 7px 0;overflow-wrap:anywhere}'
        '.widget-radio-box input{flex:0 0 auto;margin:3px 0 0 0}'
        '.jp-RenderedText pre,.widget-output pre{word-break:normal !important;overflow-wrap:anywhere;white-space:pre-wrap !important}'
        '.widget-html-content{white-space:normal;line-height:1.4}'
        '.widget-text{max-width:100%}'
        '</style>'))
except Exception:
    pass
if not getattr(plt, '_dos_paneles_br', False):
    def _show_br(*a, _orig=plt.show, **k):
        # Figura de dos paneles lado a lado: se entrega como dos imágenes que se apilan si la pantalla es angosta.
        # El PNG completo viaja en el mismo paquete (para Colab sin HTML, miniaturas y pruebas).
        import io as _io, base64 as _b64
        from matplotlib.transforms import Bbox as _Bbox
        for _num in list(plt.get_fignums()):
            fig = plt.figure(_num)
            try:
                grupos = {}
                for ax in fig.axes: grupos.setdefault(round(ax.get_position().x0, 2), []).append(ax)
                if len(grupos) != 2 or len({round(ax.get_position().y0, 2) for ax in fig.axes}) != 1: continue
                fig.canvas.draw(); r = fig.canvas.get_renderer()
                def _png(bb='tight', dpi=None):
                    b = _io.BytesIO(); fig.savefig(b, format='png', bbox_inches=bb, dpi=dpi); return _b64.b64encode(b.getvalue()).decode('ascii')
                partes = []
                for x0 in sorted(grupos):
                    bb = _Bbox.union([ax.get_tightbbox(r) for ax in grupos[x0]]).transformed(fig.dpi_scale_trans.inverted())
                    partes.append(_png(bb.padded(0.06), 144))
                html = ('<div style="display:flex;flex-wrap:wrap;gap:8px 16px;align-items:flex-start">'
                        + ''.join(f'<img alt="Esquema del punto (panel {i + 1} de 2)" src="data:image/png;base64,{p}" '
                                  'style="flex:1 1 300px;min-width:0;max-width:min(100%,420px);height:auto">' for i, p in enumerate(partes)) + '</div>')
                display({'text/html': html, 'image/png': _png(), 'text/plain': '<Figure>'}, raw=True)
                plt.close(fig)
            except Exception:
                pass
        return _orig(*a, **k)
    plt.show = _show_br; plt._dos_paneles_br = True
# --- fin pantalla angosta ---
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))


def _num_coma(s):
    import re as _re
    # reemplaza punto decimal por coma en números dentro de un string ya formateado
    return _re.sub(r"(\d)\.(\d)", r"\1,\2", s)
def print_es(*args, **kwargs):
    args2=[_num_coma(a) if isinstance(a,str) else a for a in args]
    print(*args2, **kwargs)

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

try:
    print_es
except NameError:
    def print_es(*args, **kwargs):
        import re as _re
        print(*[_re.sub(r"(\d)\.(\d)", r"\1,\2", a) if isinstance(a, str) else a for a in args], **kwargs)
# Signo menos tipográfico (−) en todo texto impreso: «-5» → «−5» (no toca guiones entre palabras, rangos ni 1e-3)
import builtins as _bi_br
if not getattr(_bi_br.print, '_menos_br', False):
    def _print_br(*a, _orig=_bi_br.print, **k):
        import re as _re
        _orig(*[_re.sub(r'(?<![\w.,\-])-(?=\d)', '−', x) if isinstance(x, str) else x for x in a], **k)
    _print_br._menos_br = True
    _bi_br.print = _print_br
# --- Cuestionarios Física BR v3: opciones barajadas, crédito por intento, nota sobre el total ---
import random as _rnd_br
try:
    PESOS_BR
except NameError:
    PESOS_BR = {}      # id_preg -> peso de cada pregunta mostrada
try:
    INTENTOS_BR
except NameError:
    INTENTOS_BR = {}   # id_preg -> [intentos, resuelta]
_CREDITO_BR = {1: 1.0, 2: 0.5, 3: 0.25}   # crédito según el intento en que se acierta
_MAX_INT_BR = 3

def _evaluar_intento(id_preg, ok, peso, explicacion, msg_ok, msg_mal):
    est = INTENTOS_BR.setdefault(id_preg, [0, False])
    if est[1]:
        print("✔ Ya resolviste esta pregunta; tu crédito quedó registrado.")
        if explicacion: print_es("→ " + explicacion)
        return
    if est[0] >= _MAX_INT_BR:
        print("⛔ Agotaste los 3 intentos con crédito: esta pregunta ya no suma.")
        if explicacion: print_es("→ " + explicacion)
        return
    est[0] += 1
    n = est[0]
    if ok:
        est[1] = True
        cred = _CREDITO_BR.get(n, 0.0)
        REGISTRO_BR[id_preg] = (cred, float(peso))
        print_es(msg_ok + f"  (intento {n}: crédito {int(cred*100)} %)")
        if explicacion: print_es("→ " + explicacion)
    else:
        REGISTRO_BR.setdefault(id_preg, (0.0, float(peso)))
        if n < _MAX_INT_BR:
            print(msg_mal + f"  Intento {n} de {_MAX_INT_BR}. Revisa tu razonamiento antes de volver a intentar.")
        else:
            print(msg_mal + " Agotaste los 3 intentos: esta pregunta ya no suma. Estudia la explicación:")
            if explicacion: print_es("→ " + explicacion)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    orden = list(range(len(opciones))); _rnd_br.shuffle(orden)   # orden distinto en cada sesión
    correcta = orden.index(indice_correcto)
    letras = "abcdefgh"
    etiquetas = [f"{letras[i]}) {opciones[j]}" for i, j in enumerate(orden)]
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=etiquetas, value=None, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            if radio.index is None:
                print("⚠ Elige una opción."); return
            _evaluar_intento(id_preg, radio.index == correcta, peso, explicacion,
                             "✅ ¡Correcto!", "❌ Incorrecto.")
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.strip().replace(",", "."))
            except ValueError: print("⚠ Escribe un número (usa coma o punto decimal)."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            _evaluar_intento(id_preg, ok, peso, explicacion,
                             f"✅ ¡Correcto!  ({valor_correcto:.3g} {unidad})", "❌ Fuera de rango.")
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not PESOS_BR:
                print("No se encontraron cuestionarios en este taller."); return
            peso_total = sum(PESOS_BR.values())
            peso_ok = sum(float(c)*p for c, p in REGISTRO_BR.values())
            nota = 5.0*peso_ok/peso_total if peso_total > 0 else 0.0
            total = len(PESOS_BR)
            resp = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0])[0] > 0)
            prim = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0, False])[1] and INTENTOS_BR[k][0] == 1)
            print("="*46)
            print("          CALIFICACIÓN FORMATIVA")
            print("="*46)
            print(f"  Preguntas respondidas:       {resp} de {total}")
            print(f"  Acertadas al primer intento: {prim}")
            print_es(f"  Puntaje ponderado:           {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*46)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*46)
            if resp < total: print(f"  ⚠ Faltan {total-resp} preguntas: cuentan como 0.")
            if nota >= 3.5: print("  ✅ ¡Buen trabajo! Dominas los conceptos.")
            elif nota >= 3.0: print("  🟡 Vas bien, repasa lo que fallaste.")
            else: print("  🔴 Repasa la teoría y vuelve a los puntos.")
            print("\n  Crédito por pregunta: 100 % al primer intento,")
            print("  50 % al segundo, 25 % al tercero. Predice y")
            print("  razona ANTES de verificar: adivinar cuesta nota.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>📊 Tu calificación del taller</h3>"
        "<i>La nota se calcula sobre TODAS las preguntas del taller y premia acertar al primer intento.</i>"),
        boton, salida]))


try:
    import matplotlib.pyplot as _plt_style
    _plt_style.rcParams['axes.spines.top']=False
    _plt_style.rcParams['axes.spines.right']=False
    _plt_style.rcParams['axes.titlesize']=12
    _plt_style.rcParams['axes.titleweight']='bold'
    _plt_style.rcParams['axes.titlecolor']='#2c3e50'
    _plt_style.rcParams['axes.labelsize']=10
    _plt_style.rcParams['axes.labelcolor']='#34495e'
    _plt_style.rcParams['font.size']=10
    _plt_style.rcParams['axes.grid']=True
    _plt_style.rcParams['grid.alpha']=0.25
    _plt_style.rcParams['grid.linestyle']='--'
    _plt_style.rcParams['axes.edgecolor']='#7f8c8d'
    _plt_style.rcParams['xtick.color']='#34495e'
    _plt_style.rcParams['ytick.color']='#34495e'
    _plt_style.rcParams['figure.facecolor']='white'
    _plt_style.rcParams['axes.facecolor']='#fcfcfd'
except Exception:
    pass
# --- fin estilo ---


def _c3(x):
    # 3 cifras significativas conservando ceros finales (8,00 y no 8); notación ×10ⁿ desde 1000
    if x != 0 and abs(x) >= 1000: return _sci(x)
    t = f"{x:#.3g}"
    return t[:-1] if t.endswith('.') else t

def _sci(x):
    e = int(np.floor(np.log10(abs(x)))); m = x/10**e
    if round(m, 2) >= 10: m /= 10; e += 1
    return f"{m:.2f}×10" + str(e).translate(str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹'))

def _3c(x):
    # 3 cifras significativas con coma decimal; ×10ⁿ fuera de 0,001 ≤ |x| < 1000 (textos de gráficas)
    import math
    try: x = float(x)
    except (TypeError, ValueError): return str(x)
    if x == 0 or not math.isfinite(x): return str(x).replace('.', ',') if x else '0'
    x = float(f"{x:.2e}")
    if abs(x) >= 1000 or abs(x) < 0.001:
        e = int(math.floor(math.log10(abs(x)))); m = x / 10**e
        if round(abs(m), 2) >= 10: m /= 10; e += 1
        return f"{m:.2f}".replace('.', ',').replace('-', '−') + '×10' + str(e).translate(str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹'))
    t = f"{x:#.3g}"
    return (t[:-1] if t.endswith('.') else t).replace('.', ',').replace('-', '−')

def esquema():
    w = 2*np.pi*60; t = np.linspace(0, 2/60, 600)
    V0 = 120*np.sqrt(2); I0 = 25.0*np.sqrt(2)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.8))
    a1.plot(t*1000, V0*np.sin(w*t), color='#2980b9', lw=2, label='v(t)')
    a1.axhline(V0, ls='--', color='#7f8c8d', lw=1); a1.axhline(120, ls=':', color='#c0392b', lw=2)
    a1.text(0.3, V0-16, 'pico V₀', fontsize=8, color='#7f8c8d'); a1.text(0.3, 128, 'eficaz V_rms', fontsize=8, color='#c0392b')
    a1.set_xlabel('t (ms)'); a1.set_ylabel('voltaje (V)'); a1.set_title('El voltaje oscila; el multímetro marca el eficaz', fontsize=10)
    p = V0*I0*np.sin(w*t)**2
    a2.plot(t*1000, p/1000, color='#e67e22', lw=2); a2.axhline(3.0, ls='--', color='#27ae60', lw=2)
    a2.text(0.3, 3.15, 'potencia media P', fontsize=8, color='#27ae60')
    a2.set_xlabel('t (ms)'); a2.set_ylabel('p(t) (kW)'); a2.set_title('Potencia instantánea: oscila entre 0 y el pico', fontsize=10)
    plt.tight_layout(); plt.show()
esquema()

def ducha(V=120.0, P_kW=3.00):
    w = 2*np.pi*60; V0 = np.sqrt(2)*V; I = P_kW*1000/V; R = V/I; I0 = np.sqrt(2)*I
    print_es(f"V₀ = {_c3(V0)} V   |   I_rms = {_c3(I)} A   |   I₀ = {_c3(I0)} A   |   R = {_c3(R)} Ω")
    print_es(f"p(t) oscila entre 0 y {_c3(V0*I0/1000)} kW; su promedio es {_c3(P_kW)} kW")
    t = np.linspace(0, 2/60, 600); p = V0*I0*np.sin(w*t)**2
    plt.figure(figsize=(8, 3.3)); plt.plot(t*1000, p/1000, color='#e67e22', lw=2)
    plt.axhline(P_kW, ls='--', color='#27ae60', lw=2, label=f'media = {_3c(P_kW)} kW')
    plt.xlabel('t (ms)'); plt.ylabel('p(t) (kW)'); plt.legend(fontsize=8); plt.title('Potencia instantánea y potencia media', fontsize=10)
    plt.tight_layout(); plt.show()

def solucion():
    V = 120.0; P = 3000.0; V0 = np.sqrt(2)*V; I = P/V; R = V/I; I0 = np.sqrt(2)*I
    print_es(f"a) V₀ = √2·V_rms = √2·120 = {_c3(V0)} V de pico")
    print_es(f"b) I_rms = P/V_rms = 3,00×10³/120 = {_c3(I)} A ;  R = V_rms/I_rms = 120/25,0 = {_c3(R)} Ω")
    print_es(f"c) I₀ = √2·I_rms = {_c3(I0)} A ;  potencia pico = V₀·I₀ = {_c3(V0)}·{_c3(I0)} = {_c3(V0*I0/1000)} kW")
    print_es(f"   La potencia media es la mitad del pico: P = V₀·I₀/2 = V_rms·I_rms = {_c3(P/1000)} kW")
    print_es("d) p(t) = V₀·I₀·sen²(ωt) = (V₀·I₀/2)·[1 − cos(2ωt)]: oscila a 2f = 120 Hz.")
    t = np.linspace(0, 1/60, 200001); pm = np.mean(V0*I0*np.sin(2*np.pi*60*t)**2)
    print_es(f"Verificación: promedio numérico de p(t) en un periodo = {_c3(pm/1000)} kW ✓ (= V_rms·I_rms)")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 1 ---")
quiz_numerico("Voltaje de pico V₀ de un tomacorriente de 120 V (rms):",
    170, 0.02, "V", "V₀=√2·120=170 V.", id_preg="CA_p1_1", peso=2.0)
quiz_numerico("Corriente eficaz que toma la ducha de 3,00 kW a 120 V (en A):",
    25.0, 0.03, "A", "I_rms=P/V_rms=3,00×10³/120=25,0 A.", id_preg="CA_p1_2", peso=2.0)
quiz_opcion_multiple("¿Con qué frecuencia oscila la potencia instantánea p(t) en la resistencia de la ducha?",
    ['120 Hz: p ∝ sen²(ωt), que se repite dos veces por cada ciclo del voltaje',
     '60,0 Hz: la potencia sigue exactamente al voltaje de la red',
     '0 Hz: en una resistencia la potencia es constante en el tiempo',
     '30,0 Hz: la potencia solo es positiva en medio ciclo de cada dos'], 0,
    "sen²(ωt) = [1 − cos(2ωt)]/2: la potencia oscila al doble de la frecuencia de la red y nunca es negativa en una resistencia.",
    id_preg="CA_p1_3", peso=2.0)
quiz_opcion_multiple("DEFIENDE TU CONCLUSIÓN: un estudiante calcula la potencia de la ducha como P = V₀·I₀ = 170·35,4 = 6,00 kW, 'porque hay que usar los valores máximos para no quedarse corto'. ¿Correcto o solo convincente?",
    ['Solo convincente: V₀·I₀ es el pico de p(t); la potencia media, que calienta el agua, es V_rms·I_rms = 3,00 kW',
     'Correcto: para dimensionar con seguridad siempre se usa la potencia máxima instantánea de la carga',
     'Correcto: la placa de 3,00 kW está en valor eficaz, y la potencia real es el doble de ese valor',
     'Solo convincente: hay que usar V₀ con I_rms, que da 4,24 kW como potencia media de la ducha'], 0,
    'La energía que calienta el agua depende del promedio de p(t), que es la mitad del pico. La opción «V₀ con I_rms» mezcla un pico con un eficaz: no corresponde a ninguna potencia física de la ducha.',
    id_preg='CA_p1_concep_1', peso=2.0)

def _interact_coma(fn, _crudos=(), **controles):
    # Como interact, con lectura de sliders en coma decimal. La salida NO se captura con «with salida:»:
    # se calculan el texto y las figuras y se escriben con append_stdout/append_display_data
    # (evita bloqueos de Voilà en Binder y funciona también en JupyterLite).
    import ipywidgets as _w, io as _io, contextlib as _cl
    import matplotlib.pyplot as _plt
    def _fmt(v, paso, crudo=False):
        # Lectura con coma decimal y 3 cifras significativas (×10ⁿ fuera de 0,001 ≤ |v| < 1000).
        # IntSlider: entero exacto. «crudo»: decimales del paso (slider que representa una medición sin redondear).
        import math
        if isinstance(v, int) and not isinstance(v, bool): return str(v)
        if crudo:
            s = repr(float(paso)); dec = 0 if 'e' in s else len(s.split('.')[1].rstrip('0'))
            return f"{v:.{min(dec, 4)}f}".replace('.', ',').replace('-', '−')
        v = float(v)
        if v == 0 or not math.isfinite(v): return '0' if v == 0 else str(v)
        v = float(f"{v:.2e}")
        if abs(v) >= 1000 or abs(v) < 0.001:
            m, e = f"{v:.2e}".split('e')
            return m.replace('.', ',').replace('-', '−') + '×10' + str(int(e)).translate(str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹'))
        t = f"{v:#.3g}"
        return (t[:-1] if t.endswith('.') else t).replace('.', ',').replace('-', '−')
    filas = []
    for nombre, c in controles.items():
        if isinstance(c, (_w.FloatSlider, _w.IntSlider)):
            c.readout = False
            lab = _w.Label(layout=_w.Layout(min_width='90px'))
            def _act(*_, c=c, lab=lab, cr=(nombre in _crudos)): lab.value = _fmt(c.value, c.step, cr)
            c.observe(_act, 'value'); _act()
            filas.append(_w.HBox([c, lab]))
        else:
            filas.append(c)
    salida = _w.Output()
    def _ejecutar(*_):
        partes, buf = [], _io.StringIO()
        def _vaciar_texto():
            t = buf.getvalue()
            if t: partes.append(('t', t)); buf.seek(0); buf.truncate(0)
        def _capturar(*a, **k):
            _vaciar_texto()
            for num in _plt.get_fignums():
                b = _io.BytesIO(); _plt.figure(num).savefig(b, format='png', bbox_inches='tight')
                partes.append(('i', b.getvalue()))
            _plt.close('all')
        _show = _plt.show; _plt.show = _capturar
        try:
            with _cl.redirect_stdout(buf):
                try: fn(**{k: c.value for k, c in controles.items()})
                except Exception as e: print('⚠ No se pudo calcular con estos valores:', e)
            _capturar()
        finally:
            _plt.show = _show
        _vaciar_texto()
        # Reemplazo en UNA sola asignación: la salida nunca queda vacía (sin saltos de la página)
        import base64 as _b64
        nuevas = []
        for tipo, val in partes:
            if tipo == 't': nuevas.append({'output_type': 'stream', 'name': 'stdout', 'text': val})
            else: nuevas.append({'output_type': 'display_data', 'metadata': {},
                                 'data': {'image/png': _b64.b64encode(val).decode('ascii'), 'text/plain': '<Figure>'}})
        salida.outputs = tuple(nuevas)
    for c in controles.values():
        c.observe(_ejecutar, 'value')
    _ejecutar()
    display(_w.VBox(filas + [salida]))

print("\n--- 🔬 Explora después de responder ---")
print("Los controles NO arrancan en los datos del enunciado. Primero resuelve y responde;\nluego explora tendencias, casos límite y comprueba tu razonamiento.")
_interact_coma(ducha,
    V=FloatSlider(value=150.0,min=100,max=240,step=10,description='V_rms (V)'),
    P_kW=FloatSlider(value=4.5,min=1,max=6,step=0.5,description='P (kW)'));

---
## Punto 2 — Reactor de línea de un variador: reactancia e impedancia (Ing. Eléctrica/Electrónica de potencia)  ★★★  [UCM, PRC]
Antes de un variador de frecuencia se instala un **reactor** (una bobina con núcleo de hierro) para suavizar la
corriente. El reactor tiene **L = 80,0 mH** y su alambre una resistencia de solo **R = 1,50 Ω**. Se conecta a
**120 V (rms), 60,0 Hz** (modelo R–L en serie, datos didácticos).

**Reactancia inductiva:**  X_L = ω·L ,  ω = 2π·f

**Impedancia:**  Z = √(R² + X_L²)

**Corriente:**  I_rms = V_rms/Z

**Desfase:**  tan φ = X_L/R  (la corriente se atrasa φ respecto al voltaje)

**Potencia que se disipa:**  P = I²·R = V·I·cos φ

> 🔧 **Aplicación en ingeniería:** la reactancia es la razón por la que bobinas, reactores, motores y transformadores
> pueden tener una resistencia de alambre muy baja sin que circule una corriente enorme: en AC el flujo variable
> induce una FEM que se opone (ley de Faraday, Inducción I). Por lo mismo, conectar por error un devanado a
> corriente continua del mismo voltaje lo quema: sin ω no hay reactancia y solo queda la resistencia del alambre.

> 🤖 **TIP-IA — Abogado del diablo:**
> Dile a un asistente: *"Voy a sostener que un reactor de 1,50 Ω conectado a 120 V deja pasar 80,0 A, porque la ley de Ohm dice I = V/R y eso no cambia en corriente alterna. Dame el mejor contraargumento y pregúntame qué papel juega la frecuencia. No me des los números."* Primero razona tú.


In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as _tk_br
if not getattr(_tk_br.ScalarFormatter, '_coma_br', False):
    # coma decimal en las marcas de los ejes y en el texto de desplazamiento (offset)
    _fmt_br_call, _fmt_br_off = _tk_br.ScalarFormatter.__call__, _tk_br.ScalarFormatter.get_offset
    _tk_br.ScalarFormatter.__call__ = lambda self, x, pos=None: _fmt_br_call(self, x, pos).replace('.', ',')
    def _off_br(self):
        import re as _re_br
        sup = str.maketrans('−-+0123456789', '⁻⁻⁺⁰¹²³⁴⁵⁶⁷⁸⁹')
        t = _fmt_br_off(self).replace('.', ',')
        t = _re_br.sub(r'^1e([−+-]?\d+)', lambda m: '×10' + m.group(1).translate(sup), t)
        return _re_br.sub(r'(\d)e([−+-]?\d+)', lambda m: m.group(1) + '×10' + m.group(2).translate(sup), t)
    _tk_br.ScalarFormatter.get_offset = _off_br
    _tk_br.ScalarFormatter._coma_br = True
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
# --- pantalla angosta (celular) ---
try:
    from IPython.display import HTML as _HTML_br
    display(_HTML_br('<style>'
        '.widget-radio{width:100% !important;max-width:100%;height:auto !important}'
        '.widget-radio-box{width:100%;max-width:100%}'
        '.widget-radio-box label{display:flex !important;flex-direction:row-reverse;justify-content:flex-end;align-items:flex-start;gap:8px;'
        'white-space:normal !important;height:auto !important;line-height:1.35 !important;margin:3px 0 7px 0;overflow-wrap:anywhere}'
        '.widget-radio-box input{flex:0 0 auto;margin:3px 0 0 0}'
        '.jp-RenderedText pre,.widget-output pre{word-break:normal !important;overflow-wrap:anywhere;white-space:pre-wrap !important}'
        '.widget-html-content{white-space:normal;line-height:1.4}'
        '.widget-text{max-width:100%}'
        '</style>'))
except Exception:
    pass
if not getattr(plt, '_dos_paneles_br', False):
    def _show_br(*a, _orig=plt.show, **k):
        # Figura de dos paneles lado a lado: se entrega como dos imágenes que se apilan si la pantalla es angosta.
        # El PNG completo viaja en el mismo paquete (para Colab sin HTML, miniaturas y pruebas).
        import io as _io, base64 as _b64
        from matplotlib.transforms import Bbox as _Bbox
        for _num in list(plt.get_fignums()):
            fig = plt.figure(_num)
            try:
                grupos = {}
                for ax in fig.axes: grupos.setdefault(round(ax.get_position().x0, 2), []).append(ax)
                if len(grupos) != 2 or len({round(ax.get_position().y0, 2) for ax in fig.axes}) != 1: continue
                fig.canvas.draw(); r = fig.canvas.get_renderer()
                def _png(bb='tight', dpi=None):
                    b = _io.BytesIO(); fig.savefig(b, format='png', bbox_inches=bb, dpi=dpi); return _b64.b64encode(b.getvalue()).decode('ascii')
                partes = []
                for x0 in sorted(grupos):
                    bb = _Bbox.union([ax.get_tightbbox(r) for ax in grupos[x0]]).transformed(fig.dpi_scale_trans.inverted())
                    partes.append(_png(bb.padded(0.06), 144))
                html = ('<div style="display:flex;flex-wrap:wrap;gap:8px 16px;align-items:flex-start">'
                        + ''.join(f'<img alt="Esquema del punto (panel {i + 1} de 2)" src="data:image/png;base64,{p}" '
                                  'style="flex:1 1 300px;min-width:0;max-width:min(100%,420px);height:auto">' for i, p in enumerate(partes)) + '</div>')
                display({'text/html': html, 'image/png': _png(), 'text/plain': '<Figure>'}, raw=True)
                plt.close(fig)
            except Exception:
                pass
        return _orig(*a, **k)
    plt.show = _show_br; plt._dos_paneles_br = True
# --- fin pantalla angosta ---
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))


def _num_coma(s):
    import re as _re
    # reemplaza punto decimal por coma en números dentro de un string ya formateado
    return _re.sub(r"(\d)\.(\d)", r"\1,\2", s)
def print_es(*args, **kwargs):
    args2=[_num_coma(a) if isinstance(a,str) else a for a in args]
    print(*args2, **kwargs)

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

try:
    print_es
except NameError:
    def print_es(*args, **kwargs):
        import re as _re
        print(*[_re.sub(r"(\d)\.(\d)", r"\1,\2", a) if isinstance(a, str) else a for a in args], **kwargs)
# Signo menos tipográfico (−) en todo texto impreso: «-5» → «−5» (no toca guiones entre palabras, rangos ni 1e-3)
import builtins as _bi_br
if not getattr(_bi_br.print, '_menos_br', False):
    def _print_br(*a, _orig=_bi_br.print, **k):
        import re as _re
        _orig(*[_re.sub(r'(?<![\w.,\-])-(?=\d)', '−', x) if isinstance(x, str) else x for x in a], **k)
    _print_br._menos_br = True
    _bi_br.print = _print_br
# --- Cuestionarios Física BR v3: opciones barajadas, crédito por intento, nota sobre el total ---
import random as _rnd_br
try:
    PESOS_BR
except NameError:
    PESOS_BR = {}      # id_preg -> peso de cada pregunta mostrada
try:
    INTENTOS_BR
except NameError:
    INTENTOS_BR = {}   # id_preg -> [intentos, resuelta]
_CREDITO_BR = {1: 1.0, 2: 0.5, 3: 0.25}   # crédito según el intento en que se acierta
_MAX_INT_BR = 3

def _evaluar_intento(id_preg, ok, peso, explicacion, msg_ok, msg_mal):
    est = INTENTOS_BR.setdefault(id_preg, [0, False])
    if est[1]:
        print("✔ Ya resolviste esta pregunta; tu crédito quedó registrado.")
        if explicacion: print_es("→ " + explicacion)
        return
    if est[0] >= _MAX_INT_BR:
        print("⛔ Agotaste los 3 intentos con crédito: esta pregunta ya no suma.")
        if explicacion: print_es("→ " + explicacion)
        return
    est[0] += 1
    n = est[0]
    if ok:
        est[1] = True
        cred = _CREDITO_BR.get(n, 0.0)
        REGISTRO_BR[id_preg] = (cred, float(peso))
        print_es(msg_ok + f"  (intento {n}: crédito {int(cred*100)} %)")
        if explicacion: print_es("→ " + explicacion)
    else:
        REGISTRO_BR.setdefault(id_preg, (0.0, float(peso)))
        if n < _MAX_INT_BR:
            print(msg_mal + f"  Intento {n} de {_MAX_INT_BR}. Revisa tu razonamiento antes de volver a intentar.")
        else:
            print(msg_mal + " Agotaste los 3 intentos: esta pregunta ya no suma. Estudia la explicación:")
            if explicacion: print_es("→ " + explicacion)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    orden = list(range(len(opciones))); _rnd_br.shuffle(orden)   # orden distinto en cada sesión
    correcta = orden.index(indice_correcto)
    letras = "abcdefgh"
    etiquetas = [f"{letras[i]}) {opciones[j]}" for i, j in enumerate(orden)]
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=etiquetas, value=None, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            if radio.index is None:
                print("⚠ Elige una opción."); return
            _evaluar_intento(id_preg, radio.index == correcta, peso, explicacion,
                             "✅ ¡Correcto!", "❌ Incorrecto.")
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.strip().replace(",", "."))
            except ValueError: print("⚠ Escribe un número (usa coma o punto decimal)."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            _evaluar_intento(id_preg, ok, peso, explicacion,
                             f"✅ ¡Correcto!  ({valor_correcto:.3g} {unidad})", "❌ Fuera de rango.")
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not PESOS_BR:
                print("No se encontraron cuestionarios en este taller."); return
            peso_total = sum(PESOS_BR.values())
            peso_ok = sum(float(c)*p for c, p in REGISTRO_BR.values())
            nota = 5.0*peso_ok/peso_total if peso_total > 0 else 0.0
            total = len(PESOS_BR)
            resp = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0])[0] > 0)
            prim = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0, False])[1] and INTENTOS_BR[k][0] == 1)
            print("="*46)
            print("          CALIFICACIÓN FORMATIVA")
            print("="*46)
            print(f"  Preguntas respondidas:       {resp} de {total}")
            print(f"  Acertadas al primer intento: {prim}")
            print_es(f"  Puntaje ponderado:           {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*46)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*46)
            if resp < total: print(f"  ⚠ Faltan {total-resp} preguntas: cuentan como 0.")
            if nota >= 3.5: print("  ✅ ¡Buen trabajo! Dominas los conceptos.")
            elif nota >= 3.0: print("  🟡 Vas bien, repasa lo que fallaste.")
            else: print("  🔴 Repasa la teoría y vuelve a los puntos.")
            print("\n  Crédito por pregunta: 100 % al primer intento,")
            print("  50 % al segundo, 25 % al tercero. Predice y")
            print("  razona ANTES de verificar: adivinar cuesta nota.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>📊 Tu calificación del taller</h3>"
        "<i>La nota se calcula sobre TODAS las preguntas del taller y premia acertar al primer intento.</i>"),
        boton, salida]))


try:
    import matplotlib.pyplot as _plt_style
    _plt_style.rcParams['axes.spines.top']=False
    _plt_style.rcParams['axes.spines.right']=False
    _plt_style.rcParams['axes.titlesize']=12
    _plt_style.rcParams['axes.titleweight']='bold'
    _plt_style.rcParams['axes.titlecolor']='#2c3e50'
    _plt_style.rcParams['axes.labelsize']=10
    _plt_style.rcParams['axes.labelcolor']='#34495e'
    _plt_style.rcParams['font.size']=10
    _plt_style.rcParams['axes.grid']=True
    _plt_style.rcParams['grid.alpha']=0.25
    _plt_style.rcParams['grid.linestyle']='--'
    _plt_style.rcParams['axes.edgecolor']='#7f8c8d'
    _plt_style.rcParams['xtick.color']='#34495e'
    _plt_style.rcParams['ytick.color']='#34495e'
    _plt_style.rcParams['figure.facecolor']='white'
    _plt_style.rcParams['axes.facecolor']='#fcfcfd'
except Exception:
    pass
# --- fin estilo ---


def _c3(x):
    # 3 cifras significativas conservando ceros finales (8,00 y no 8); notación ×10ⁿ desde 1000
    if x != 0 and abs(x) >= 1000: return _sci(x)
    t = f"{x:#.3g}"
    return t[:-1] if t.endswith('.') else t

def _sci(x):
    e = int(np.floor(np.log10(abs(x)))); m = x/10**e
    if round(m, 2) >= 10: m /= 10; e += 1
    return f"{m:.2f}×10" + str(e).translate(str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹'))

def _3c(x):
    # 3 cifras significativas con coma decimal; ×10ⁿ fuera de 0,001 ≤ |x| < 1000 (textos de gráficas)
    import math
    try: x = float(x)
    except (TypeError, ValueError): return str(x)
    if x == 0 or not math.isfinite(x): return str(x).replace('.', ',') if x else '0'
    x = float(f"{x:.2e}")
    if abs(x) >= 1000 or abs(x) < 0.001:
        e = int(math.floor(math.log10(abs(x)))); m = x / 10**e
        if round(abs(m), 2) >= 10: m /= 10; e += 1
        return f"{m:.2f}".replace('.', ',').replace('-', '−') + '×10' + str(e).translate(str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹'))
    t = f"{x:#.3g}"
    return (t[:-1] if t.endswith('.') else t).replace('.', ',').replace('-', '−')

def esquema():
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.8), gridspec_kw={'width_ratios': [1, 1.2]})
    # circuito R–L con fuente AC
    a1.add_patch(plt.Circle((0.6, 1.5), 0.35, fill=False, ec='#2c3e50', lw=2)); a1.text(0.6, 1.5, '~', ha='center', va='center', fontsize=18)
    a1.plot([0.6, 0.6, 3.4, 3.4, 0.6, 0.6], [1.85, 2.6, 2.6, 0.4, 0.4, 1.15], color='#2c3e50', lw=1.5)
    xs = np.linspace(1.2, 2.0, 7); a1.plot(xs, 2.6 + 0.12*np.array([0, 1, -1, 1, -1, 1, 0]), color='#c0392b', lw=2); a1.text(1.6, 2.85, 'R', color='#c0392b', ha='center')
    th = np.linspace(0, 6*np.pi, 300); a1.plot(3.4 + 0.12*np.sin(th), 2.2 - th/(6*np.pi)*1.2, color='#2980b9', lw=2); a1.text(3.7, 1.6, 'L', color='#2980b9')
    a1.text(0.6, 0.0, '120 V · 60,0 Hz', ha='center', fontsize=8); a1.set_xlim(0, 4.2); a1.set_ylim(-0.3, 3.1); a1.axis('off'); a1.set_aspect('equal')
    a1.set_title('Reactor: R en serie con L', fontsize=10)
    # triángulo de impedancia
    R, XL = 1.5, 2*np.pi*60*0.08
    a2.annotate('', xy=(R, 0), xytext=(0, 0), arrowprops=dict(arrowstyle='-|>', color='#c0392b', lw=2))
    a2.annotate('', xy=(R, XL), xytext=(R, 0), arrowprops=dict(arrowstyle='-|>', color='#2980b9', lw=2))
    a2.annotate('', xy=(R, XL), xytext=(0, 0), arrowprops=dict(arrowstyle='-|>', color='#8e44ad', lw=2.5))
    a2.text(R/2, -2.5, 'R', color='#c0392b', ha='center'); a2.text(R + 0.4, XL/2, 'X_L = ωL', color='#2980b9')
    a2.text(-1.2, XL/2, 'Z', color='#8e44ad', fontsize=12); a2.text(0.25, 3.5, 'φ', fontsize=11)
    a2.set_xlim(-3, 8); a2.set_ylim(-4, 34); a2.set_xlabel('parte resistiva (Ω)'); a2.set_ylabel('parte reactiva (Ω)')
    a2.set_title('Triángulo de impedancia (casi vertical: φ cercano a 90°)', fontsize=10)
    plt.tight_layout(); plt.show()
esquema()

def reactor(R=1.5, L_mH=80.0, f=60.0, V=120.0):
    w = 2*np.pi*f; XL = w*L_mH/1000; Z = np.hypot(R, XL); I = V/Z; phi = np.degrees(np.arctan2(XL, R))
    print_es(f"X_L = {_c3(XL)} Ω   |   Z = {_c3(Z)} Ω   |   I_rms = {_c3(I)} A   |   φ = {_c3(phi)}°   |   FP = {_c3(R/Z)}")
    print_es(f"Potencia disipada P = I²·R = {_c3(I**2*R)} W   |   en DC (f = 0): I = V/R = {_c3(V/R)} A")
    fs = np.linspace(0, 120, 300); Is = V/np.hypot(R, 2*np.pi*fs*L_mH/1000)
    plt.figure(figsize=(8, 3.3)); plt.semilogy(fs, Is, color='#8e44ad', lw=2); plt.scatter([f], [I], color='#c0392b', zorder=5)
    import matplotlib.ticker as _tk; _ax = plt.gca(); _ax.yaxis.set_major_locator(_tk.LogLocator(subs=(1, 2, 5))); _ax.yaxis.set_major_formatter(_tk.FuncFormatter(lambda y, _: f'{y:g}'.replace('.', ','))); _ax.yaxis.set_minor_formatter(_tk.NullFormatter())
    plt.xlabel('frecuencia f (Hz)'); plt.ylabel('I_rms (A, escala log)'); plt.title('La corriente cae al crecer la frecuencia (X_L = ωL)', fontsize=10)
    plt.tight_layout(); plt.show()

def solucion():
    R = 1.5; L = 0.080; V = 120.0; w = 2*np.pi*60; XL = w*L; Z = np.hypot(R, XL); I = V/Z; phi = np.degrees(np.arctan2(XL, R))
    print_es(f"a) ω = 2π·60,0 = {_c3(w)} rad/s ;  X_L = ω·L = {_c3(w)}·0,0800 = {_c3(XL)} Ω")
    print_es(f"b) Z = √(1,50² + {_c3(XL)}²) = {_c3(Z)} Ω  (la resistencia casi no pesa) ;  I_rms = 120/{_c3(Z)} = {_c3(I)} A")
    print_es(f"c) φ = arctan(X_L/R) = arctan({_c3(XL)}/1,50) = {_c3(phi)}° ;  FP = cos φ = R/Z = {_c3(R/Z)}")
    print_es(f"d) P = I²·R = {_c3(I)}²·1,50 = {_c3(I**2*R)} W: casi toda la energía va y vuelve en cada ciclo.")
    print_es(f"e) En DC: I = V/R = 120/1,50 = {_c3(V/R)} A y P = V²/R = {_c3(V**2/R/1000)} kW: el reactor se quema.")
    Ic = V/complex(R, XL)
    print_es(f"Verificación con fasores: I = V/(R + jX_L) → |I| = {_c3(abs(Ic))} A ✓ ;  V·I·cos φ = {_c3(V*abs(Ic)*R/Z)} W = I²·R ✓")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 2 ---")
quiz_numerico("Reactancia inductiva X_L del reactor de 80,0 mH a 60,0 Hz (en Ω):",
    30.2, 0.03, "Ω", "X_L=2π·60,0·0,0800=30,2 Ω.", id_preg="CA_p2_1", peso=2.0)
quiz_numerico("Corriente eficaz por el reactor conectado a 120 V (rms), 60,0 Hz (en A):",
    3.97, 0.03, "A", "Z=√(1,50²+30,2²)=30,2 Ω; I=120/30,2=3,97 A.", id_preg="CA_p2_2", peso=3.0)
quiz_opcion_multiple("PROBLEMA INESPERADO: si la frecuencia de la red bajara a la mitad (30,0 Hz) con el mismo voltaje, la corriente del reactor:",
    ['Casi se duplicaría: X_L = ωL se reduce a la mitad y domina la impedancia',
     'Se quedaría igual: la resistencia del alambre no depende de la frecuencia',
     'Bajaría a la mitad: con menos frecuencia llega menos energía por segundo',
     'Se haría infinita: a baja frecuencia la bobina se comporta como un cortocircuito'], 0,
    "Con X_L ≫ R, I ≈ V/(ωL): reducir ω a la mitad casi duplica I (7,92 A). Solo en DC (f = 0) queda únicamente R.",
    id_preg="CA_p2_3", peso=2.0)
quiz_opcion_multiple("DEFIENDE TU CONCLUSIÓN: un técnico afirma que 'este reactor funciona bien a 120 V AC, pero si alguien lo conecta a 120 V DC se quema, aunque el voltaje sea el mismo'. ¿Correcto o solo convincente?",
    ['Correcto: en DC no hay flujo variable ni reactancia; solo queda R = 1,50 Ω y la corriente sube a 80,0 A',
     'Solo convincente: el voltaje es el mismo, así que la corriente y el calentamiento también son los mismos',
     'Correcto: la corriente continua siempre es más peligrosa porque su voltaje no pasa por cero',
     'Solo convincente: en DC la bobina se comporta como un circuito abierto y la corriente sería cero'], 0,
    'Esta vez es CORRECTO. La opción «la corriente continua siempre es más peligrosa…» acierta el veredicto con una razón falsa: el problema no es el paso por cero, sino que en DC desaparece la reactancia. Circuito abierto en DC es el condensador, no la bobina.',
    id_preg='CA_p2_concep_2', peso=2.0)

def _interact_coma(fn, _crudos=(), **controles):
    # Como interact, con lectura de sliders en coma decimal. La salida NO se captura con «with salida:»:
    # se calculan el texto y las figuras y se escriben con append_stdout/append_display_data
    # (evita bloqueos de Voilà en Binder y funciona también en JupyterLite).
    import ipywidgets as _w, io as _io, contextlib as _cl
    import matplotlib.pyplot as _plt
    def _fmt(v, paso, crudo=False):
        # Lectura con coma decimal y 3 cifras significativas (×10ⁿ fuera de 0,001 ≤ |v| < 1000).
        # IntSlider: entero exacto. «crudo»: decimales del paso (slider que representa una medición sin redondear).
        import math
        if isinstance(v, int) and not isinstance(v, bool): return str(v)
        if crudo:
            s = repr(float(paso)); dec = 0 if 'e' in s else len(s.split('.')[1].rstrip('0'))
            return f"{v:.{min(dec, 4)}f}".replace('.', ',').replace('-', '−')
        v = float(v)
        if v == 0 or not math.isfinite(v): return '0' if v == 0 else str(v)
        v = float(f"{v:.2e}")
        if abs(v) >= 1000 or abs(v) < 0.001:
            m, e = f"{v:.2e}".split('e')
            return m.replace('.', ',').replace('-', '−') + '×10' + str(int(e)).translate(str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹'))
        t = f"{v:#.3g}"
        return (t[:-1] if t.endswith('.') else t).replace('.', ',').replace('-', '−')
    filas = []
    for nombre, c in controles.items():
        if isinstance(c, (_w.FloatSlider, _w.IntSlider)):
            c.readout = False
            lab = _w.Label(layout=_w.Layout(min_width='90px'))
            def _act(*_, c=c, lab=lab, cr=(nombre in _crudos)): lab.value = _fmt(c.value, c.step, cr)
            c.observe(_act, 'value'); _act()
            filas.append(_w.HBox([c, lab]))
        else:
            filas.append(c)
    salida = _w.Output()
    def _ejecutar(*_):
        partes, buf = [], _io.StringIO()
        def _vaciar_texto():
            t = buf.getvalue()
            if t: partes.append(('t', t)); buf.seek(0); buf.truncate(0)
        def _capturar(*a, **k):
            _vaciar_texto()
            for num in _plt.get_fignums():
                b = _io.BytesIO(); _plt.figure(num).savefig(b, format='png', bbox_inches='tight')
                partes.append(('i', b.getvalue()))
            _plt.close('all')
        _show = _plt.show; _plt.show = _capturar
        try:
            with _cl.redirect_stdout(buf):
                try: fn(**{k: c.value for k, c in controles.items()})
                except Exception as e: print('⚠ No se pudo calcular con estos valores:', e)
            _capturar()
        finally:
            _plt.show = _show
        _vaciar_texto()
        # Reemplazo en UNA sola asignación: la salida nunca queda vacía (sin saltos de la página)
        import base64 as _b64
        nuevas = []
        for tipo, val in partes:
            if tipo == 't': nuevas.append({'output_type': 'stream', 'name': 'stdout', 'text': val})
            else: nuevas.append({'output_type': 'display_data', 'metadata': {},
                                 'data': {'image/png': _b64.b64encode(val).decode('ascii'), 'text/plain': '<Figure>'}})
        salida.outputs = tuple(nuevas)
    for c in controles.values():
        c.observe(_ejecutar, 'value')
    _ejecutar()
    display(_w.VBox(filas + [salida]))

def _explora_sin_cifras(fn):
    # Opción 3: exploración cualitativa; las cifras de este punto solo están en «Mostrar solución».
    import functools, io, contextlib, re as _re
    _pat = _re.compile(r'(?<![\w.,/])[-−]?\d+(?:[.,]\d+)?(?:\s?×\s?10[⁻⁰¹²³⁴⁵⁶⁷⁸⁹]+|[eE][+-]?\d+)?(?![\wπ·(]|[.,]\d)')
    def _m(t): return _pat.sub('…', t)
    def _ocultar():
        for num in plt.get_fignums():
            fig = plt.figure(num)
            textos = list(fig.texts) + ([fig._suptitle] if getattr(fig, '_suptitle', None) else [])
            for ax in fig.axes:
                textos += [ax.title, ax._left_title, ax._right_title] + list(ax.texts)
                if ax.get_legend(): textos += list(ax.get_legend().get_texts())
            for t in textos: t.set_text(_m(t.get_text()))
    @functools.wraps(fn)
    def envuelta(*a, **k):
        _show = plt.show
        def _show2(*sa, **sk):
            _ocultar(); _show(*sa, **sk)
        buf = io.StringIO(); plt.show = _show2
        try:
            with contextlib.redirect_stdout(buf): fn(*a, **k)
            _ocultar()
        finally:
            plt.show = _show
        for linea in buf.getvalue().splitlines():
            print(linea if ('⚠' in linea or '⛔' in linea) else _m(linea))
        print("(Exploración cualitativa: observa tendencias. Las cifras de este punto están en «Mostrar solución».)")
    return envuelta

print("\n--- 🔬 Explora después de responder ---")
print("Los controles NO arrancan en los datos del enunciado. Primero resuelve y responde;\nluego explora tendencias, casos límite y comprueba tu razonamiento.")
_interact_coma(_explora_sin_cifras(reactor),
    R=FloatSlider(value=4.0,min=0.5,max=10,step=0.5,description='R (Ω)'),
    L_mH=FloatSlider(value=130.0,min=10,max=200,step=10,description='L (mH)'),
    f=FloatSlider(value=90.0,min=0,max=120,step=5,description='f (Hz)'),
    V=FloatSlider(value=150.0,min=60,max=240,step=10,description='V_rms (V)'));

---
## Punto 3 — Trilladora de café: potencias activa, reactiva y aparente (Ing. Eléctrica/Agroindustrial)  ★★★  [PRC, ULC]
Una trilladora de café alimentada a **240 V (rms), 60,0 Hz** mueve sus máquinas con motores de inducción. La carga
total consume **P = 18,0 kW** con un factor de potencia **FP = 0,680 en atraso** (inductivo). El alimentador
desde el transformador tiene una resistencia (ida y vuelta) de **R_línea = 0,0500 Ω**. Modelo monofásico
equivalente con datos didácticos.

**Potencias:**  S = P/FP ,  I = S/V ,  Q = P·tan φ = √(S² − P²)

**Pérdidas en el alimentador:**  P_pérd = I²·R_línea

**Regulación (Res. CREG 015 de 2018, art. 16 y cap. 12; Res. CREG 101 035 de 2024):** el FP inductivo debe ser ≥ 0,90,
es decir, la energía reactiva inductiva no debe superar el **50 %** de la activa en cada periodo horario; el exceso se cobra.
(Exacto: Q/P = 0,500 ↔ FP = 1/√1,25 = 0,894; la norma lo redondea a 0,90.)

> 🔧 **Aplicación en ingeniería:** los motores necesitan potencia reactiva para crear su campo magnético; esa
> energía no hace trabajo, pero sí viaja por los cables. Con el mismo trabajo útil, un FP bajo exige más corriente:
> más pérdidas I²·R, más caída de tensión y transformadores y cables más grandes. Por eso el operador de red cobra el
> exceso de reactiva a usuarios como trilladoras, trapiches o talleres.
> *Fuentes:* Res. CREG 015 de 2018, art. 16 y cap. 12 (modificada por las Res. CREG 199 de 2019 y 195 de 2020); Res. CREG 101 035
> de 2024 (límites de FP inductivo y capacitivo, tabla FP 0,90 ↔ 50 %). Consultadas en el gestor normativo de la CREG, noviembre de 2026.

> 🤖 **TIP-IA — Verificador crítico:**
> Dile a un asistente: *"Calculé la corriente de una carga de 18,0 kW a 240 V con FP 0,680 usando I = P/V. Actúa como evaluador escéptico: pregúntame si esa es la corriente que circula por los cables o solo la parte que hace trabajo, y qué cambia con el factor de potencia. No me des el número."* Primero resuelves tú.


In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as _tk_br
if not getattr(_tk_br.ScalarFormatter, '_coma_br', False):
    # coma decimal en las marcas de los ejes y en el texto de desplazamiento (offset)
    _fmt_br_call, _fmt_br_off = _tk_br.ScalarFormatter.__call__, _tk_br.ScalarFormatter.get_offset
    _tk_br.ScalarFormatter.__call__ = lambda self, x, pos=None: _fmt_br_call(self, x, pos).replace('.', ',')
    def _off_br(self):
        import re as _re_br
        sup = str.maketrans('−-+0123456789', '⁻⁻⁺⁰¹²³⁴⁵⁶⁷⁸⁹')
        t = _fmt_br_off(self).replace('.', ',')
        t = _re_br.sub(r'^1e([−+-]?\d+)', lambda m: '×10' + m.group(1).translate(sup), t)
        return _re_br.sub(r'(\d)e([−+-]?\d+)', lambda m: m.group(1) + '×10' + m.group(2).translate(sup), t)
    _tk_br.ScalarFormatter.get_offset = _off_br
    _tk_br.ScalarFormatter._coma_br = True
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
# --- pantalla angosta (celular) ---
try:
    from IPython.display import HTML as _HTML_br
    display(_HTML_br('<style>'
        '.widget-radio{width:100% !important;max-width:100%;height:auto !important}'
        '.widget-radio-box{width:100%;max-width:100%}'
        '.widget-radio-box label{display:flex !important;flex-direction:row-reverse;justify-content:flex-end;align-items:flex-start;gap:8px;'
        'white-space:normal !important;height:auto !important;line-height:1.35 !important;margin:3px 0 7px 0;overflow-wrap:anywhere}'
        '.widget-radio-box input{flex:0 0 auto;margin:3px 0 0 0}'
        '.jp-RenderedText pre,.widget-output pre{word-break:normal !important;overflow-wrap:anywhere;white-space:pre-wrap !important}'
        '.widget-html-content{white-space:normal;line-height:1.4}'
        '.widget-text{max-width:100%}'
        '</style>'))
except Exception:
    pass
if not getattr(plt, '_dos_paneles_br', False):
    def _show_br(*a, _orig=plt.show, **k):
        # Figura de dos paneles lado a lado: se entrega como dos imágenes que se apilan si la pantalla es angosta.
        # El PNG completo viaja en el mismo paquete (para Colab sin HTML, miniaturas y pruebas).
        import io as _io, base64 as _b64
        from matplotlib.transforms import Bbox as _Bbox
        for _num in list(plt.get_fignums()):
            fig = plt.figure(_num)
            try:
                grupos = {}
                for ax in fig.axes: grupos.setdefault(round(ax.get_position().x0, 2), []).append(ax)
                if len(grupos) != 2 or len({round(ax.get_position().y0, 2) for ax in fig.axes}) != 1: continue
                fig.canvas.draw(); r = fig.canvas.get_renderer()
                def _png(bb='tight', dpi=None):
                    b = _io.BytesIO(); fig.savefig(b, format='png', bbox_inches=bb, dpi=dpi); return _b64.b64encode(b.getvalue()).decode('ascii')
                partes = []
                for x0 in sorted(grupos):
                    bb = _Bbox.union([ax.get_tightbbox(r) for ax in grupos[x0]]).transformed(fig.dpi_scale_trans.inverted())
                    partes.append(_png(bb.padded(0.06), 144))
                html = ('<div style="display:flex;flex-wrap:wrap;gap:8px 16px;align-items:flex-start">'
                        + ''.join(f'<img alt="Esquema del punto (panel {i + 1} de 2)" src="data:image/png;base64,{p}" '
                                  'style="flex:1 1 300px;min-width:0;max-width:min(100%,420px);height:auto">' for i, p in enumerate(partes)) + '</div>')
                display({'text/html': html, 'image/png': _png(), 'text/plain': '<Figure>'}, raw=True)
                plt.close(fig)
            except Exception:
                pass
        return _orig(*a, **k)
    plt.show = _show_br; plt._dos_paneles_br = True
# --- fin pantalla angosta ---
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))


def _num_coma(s):
    import re as _re
    # reemplaza punto decimal por coma en números dentro de un string ya formateado
    return _re.sub(r"(\d)\.(\d)", r"\1,\2", s)
def print_es(*args, **kwargs):
    args2=[_num_coma(a) if isinstance(a,str) else a for a in args]
    print(*args2, **kwargs)

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

try:
    print_es
except NameError:
    def print_es(*args, **kwargs):
        import re as _re
        print(*[_re.sub(r"(\d)\.(\d)", r"\1,\2", a) if isinstance(a, str) else a for a in args], **kwargs)
# Signo menos tipográfico (−) en todo texto impreso: «-5» → «−5» (no toca guiones entre palabras, rangos ni 1e-3)
import builtins as _bi_br
if not getattr(_bi_br.print, '_menos_br', False):
    def _print_br(*a, _orig=_bi_br.print, **k):
        import re as _re
        _orig(*[_re.sub(r'(?<![\w.,\-])-(?=\d)', '−', x) if isinstance(x, str) else x for x in a], **k)
    _print_br._menos_br = True
    _bi_br.print = _print_br
# --- Cuestionarios Física BR v3: opciones barajadas, crédito por intento, nota sobre el total ---
import random as _rnd_br
try:
    PESOS_BR
except NameError:
    PESOS_BR = {}      # id_preg -> peso de cada pregunta mostrada
try:
    INTENTOS_BR
except NameError:
    INTENTOS_BR = {}   # id_preg -> [intentos, resuelta]
_CREDITO_BR = {1: 1.0, 2: 0.5, 3: 0.25}   # crédito según el intento en que se acierta
_MAX_INT_BR = 3

def _evaluar_intento(id_preg, ok, peso, explicacion, msg_ok, msg_mal):
    est = INTENTOS_BR.setdefault(id_preg, [0, False])
    if est[1]:
        print("✔ Ya resolviste esta pregunta; tu crédito quedó registrado.")
        if explicacion: print_es("→ " + explicacion)
        return
    if est[0] >= _MAX_INT_BR:
        print("⛔ Agotaste los 3 intentos con crédito: esta pregunta ya no suma.")
        if explicacion: print_es("→ " + explicacion)
        return
    est[0] += 1
    n = est[0]
    if ok:
        est[1] = True
        cred = _CREDITO_BR.get(n, 0.0)
        REGISTRO_BR[id_preg] = (cred, float(peso))
        print_es(msg_ok + f"  (intento {n}: crédito {int(cred*100)} %)")
        if explicacion: print_es("→ " + explicacion)
    else:
        REGISTRO_BR.setdefault(id_preg, (0.0, float(peso)))
        if n < _MAX_INT_BR:
            print(msg_mal + f"  Intento {n} de {_MAX_INT_BR}. Revisa tu razonamiento antes de volver a intentar.")
        else:
            print(msg_mal + " Agotaste los 3 intentos: esta pregunta ya no suma. Estudia la explicación:")
            if explicacion: print_es("→ " + explicacion)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    orden = list(range(len(opciones))); _rnd_br.shuffle(orden)   # orden distinto en cada sesión
    correcta = orden.index(indice_correcto)
    letras = "abcdefgh"
    etiquetas = [f"{letras[i]}) {opciones[j]}" for i, j in enumerate(orden)]
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=etiquetas, value=None, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            if radio.index is None:
                print("⚠ Elige una opción."); return
            _evaluar_intento(id_preg, radio.index == correcta, peso, explicacion,
                             "✅ ¡Correcto!", "❌ Incorrecto.")
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.strip().replace(",", "."))
            except ValueError: print("⚠ Escribe un número (usa coma o punto decimal)."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            _evaluar_intento(id_preg, ok, peso, explicacion,
                             f"✅ ¡Correcto!  ({valor_correcto:.3g} {unidad})", "❌ Fuera de rango.")
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not PESOS_BR:
                print("No se encontraron cuestionarios en este taller."); return
            peso_total = sum(PESOS_BR.values())
            peso_ok = sum(float(c)*p for c, p in REGISTRO_BR.values())
            nota = 5.0*peso_ok/peso_total if peso_total > 0 else 0.0
            total = len(PESOS_BR)
            resp = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0])[0] > 0)
            prim = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0, False])[1] and INTENTOS_BR[k][0] == 1)
            print("="*46)
            print("          CALIFICACIÓN FORMATIVA")
            print("="*46)
            print(f"  Preguntas respondidas:       {resp} de {total}")
            print(f"  Acertadas al primer intento: {prim}")
            print_es(f"  Puntaje ponderado:           {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*46)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*46)
            if resp < total: print(f"  ⚠ Faltan {total-resp} preguntas: cuentan como 0.")
            if nota >= 3.5: print("  ✅ ¡Buen trabajo! Dominas los conceptos.")
            elif nota >= 3.0: print("  🟡 Vas bien, repasa lo que fallaste.")
            else: print("  🔴 Repasa la teoría y vuelve a los puntos.")
            print("\n  Crédito por pregunta: 100 % al primer intento,")
            print("  50 % al segundo, 25 % al tercero. Predice y")
            print("  razona ANTES de verificar: adivinar cuesta nota.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>📊 Tu calificación del taller</h3>"
        "<i>La nota se calcula sobre TODAS las preguntas del taller y premia acertar al primer intento.</i>"),
        boton, salida]))


try:
    import matplotlib.pyplot as _plt_style
    _plt_style.rcParams['axes.spines.top']=False
    _plt_style.rcParams['axes.spines.right']=False
    _plt_style.rcParams['axes.titlesize']=12
    _plt_style.rcParams['axes.titleweight']='bold'
    _plt_style.rcParams['axes.titlecolor']='#2c3e50'
    _plt_style.rcParams['axes.labelsize']=10
    _plt_style.rcParams['axes.labelcolor']='#34495e'
    _plt_style.rcParams['font.size']=10
    _plt_style.rcParams['axes.grid']=True
    _plt_style.rcParams['grid.alpha']=0.25
    _plt_style.rcParams['grid.linestyle']='--'
    _plt_style.rcParams['axes.edgecolor']='#7f8c8d'
    _plt_style.rcParams['xtick.color']='#34495e'
    _plt_style.rcParams['ytick.color']='#34495e'
    _plt_style.rcParams['figure.facecolor']='white'
    _plt_style.rcParams['axes.facecolor']='#fcfcfd'
except Exception:
    pass
# --- fin estilo ---


def _c3(x):
    # 3 cifras significativas conservando ceros finales (8,00 y no 8); notación ×10ⁿ desde 1000
    if x != 0 and abs(x) >= 1000: return _sci(x)
    t = f"{x:#.3g}"
    return t[:-1] if t.endswith('.') else t

def _sci(x):
    e = int(np.floor(np.log10(abs(x)))); m = x/10**e
    if round(m, 2) >= 10: m /= 10; e += 1
    return f"{m:.2f}×10" + str(e).translate(str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹'))

def _3c(x):
    # 3 cifras significativas con coma decimal; ×10ⁿ fuera de 0,001 ≤ |x| < 1000 (textos de gráficas)
    import math
    try: x = float(x)
    except (TypeError, ValueError): return str(x)
    if x == 0 or not math.isfinite(x): return str(x).replace('.', ',') if x else '0'
    x = float(f"{x:.2e}")
    if abs(x) >= 1000 or abs(x) < 0.001:
        e = int(math.floor(math.log10(abs(x)))); m = x / 10**e
        if round(abs(m), 2) >= 10: m /= 10; e += 1
        return f"{m:.2f}".replace('.', ',').replace('-', '−') + '×10' + str(e).translate(str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹'))
    t = f"{x:#.3g}"
    return (t[:-1] if t.endswith('.') else t).replace('.', ',').replace('-', '−')

def esquema():
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.8))
    P, fp = 18.0, 0.68; Q = P*np.tan(np.arccos(fp)); S = P/fp
    a1.annotate('', xy=(P, 0), xytext=(0, 0), arrowprops=dict(arrowstyle='-|>', color='#27ae60', lw=2.5))
    a1.annotate('', xy=(P, Q), xytext=(P, 0), arrowprops=dict(arrowstyle='-|>', color='#e67e22', lw=2.5))
    a1.annotate('', xy=(P, Q), xytext=(0, 0), arrowprops=dict(arrowstyle='-|>', color='#8e44ad', lw=2.5))
    a1.plot([0, P], [0, 0.5*P], ls=':', color='#c0392b', lw=1.5); a1.text(P*0.55, 0.5*P*0.55 - 2.2, 'límite CREG: Q = 0,500·P', fontsize=7, color='#c0392b')
    a1.text(P/2, -1.8, 'P (kW): trabajo útil', ha='center', fontsize=8, color='#27ae60')
    a1.text(P + 0.6, Q/2, 'Q (kVAR)', fontsize=8, color='#e67e22'); a1.text(P/2 - 3, Q/2 + 1.5, 'S (kVA)', fontsize=8, color='#8e44ad')
    a1.set_xlim(-1, 25); a1.set_ylim(-3, 23); a1.set_aspect('equal'); a1.axis('off'); a1.set_title('Triángulo de potencias (FP = cos φ = P/S)', fontsize=10)
    fps = np.linspace(0.5, 1.0, 200); Is = 18000/(240*fps)
    a2.plot(fps, Is, color='#8e44ad', lw=2); a2.axvline(0.894, ls='--', color='#c0392b'); a2.text(0.80, 140, 'FP 0,894', color='#c0392b', fontsize=8)
    a2.set_xlabel('factor de potencia'); a2.set_ylabel('corriente (A)'); a2.set_title('Mismo trabajo útil, más corriente si el FP baja', fontsize=10)
    plt.tight_layout(); plt.show()
esquema()

def trilladora(P_kW=18.0, FP=0.68, V=240.0, R_linea=0.05):
    P = P_kW*1000; S = P/FP; I = S/V; Q = P*np.tan(np.arccos(FP)); perd = I**2*R_linea; perd1 = (P/V)**2*R_linea
    print_es(f"S = {_c3(S/1000)} kVA   |   I = {_c3(I)} A   |   Q = {_c3(Q/1000)} kVAR   |   Q/P = {_c3(Q/P)}")
    print_es(f"Pérdidas en el alimentador = {_c3(perd)} W   (con FP = 1 serían {_c3(perd1)} W)")
    print_es("⚠ Q/P > 0,500 (FP < 0,90): hay exceso de reactiva inductiva facturable." if Q/P > 0.5 else "Q/P ≤ 0,500: sin exceso de reactiva facturable.")
    fps = np.linspace(0.5, 1.0, 200)
    plt.figure(figsize=(8, 3.3)); plt.plot(fps, ((P/(V*fps))**2*R_linea), color='#c0392b', lw=2); plt.scatter([FP], [perd], color='#2c3e50', zorder=5)
    plt.axvline(0.894, ls='--', color='#7f8c8d'); plt.xlabel('factor de potencia'); plt.ylabel('pérdidas I²R (W)')
    plt.title('Pérdidas en el alimentador: crecen al bajar el FP', fontsize=10); plt.tight_layout(); plt.show()

def solucion():
    P = 18000.0; FP = 0.680; V = 240.0; Rl = 0.05; S = P/FP; I = S/V; phi = np.arccos(FP); Q = P*np.tan(phi)
    print_es(f"a) S = P/FP = 18,0/0,680 = {_c3(S/1000)} kVA ;  I = S/V = {_c3(S)}/240 = {_c3(I)} A")
    print_es(f"   (I = P/V = 75,0 A sería la corriente con FP = 1, no la que circula)")
    print_es(f"b) φ = arccos 0,680 = {_c3(np.degrees(phi))}° ;  Q = P·tan φ = 18,0·{_c3(np.tan(phi))} = {_c3(Q/1000)} kVAR")
    print_es(f"c) Q/P = {_c3(Q/P)} > 0,500 (FP 0,680 < 0,90) → la trilladora paga exceso de reactiva inductiva (Res. CREG 015/2018 y 101 035/2024).")
    print_es(f"d) Pérdidas: I²·R = {_c3(I)}²·0,0500 = {_c3(I**2*Rl)} W ;  con FP = 1: 75,0²·0,0500 = {_c3((P/V)**2*Rl)} W")
    print_es(f"   Razón = 1/FP² = {_c3(1/FP**2)}: el FP bajo duplica las pérdidas en el cable.")
    print_es(f"Verificación: √(S² − P²) = {_c3(np.sqrt(S**2 - P**2)/1000)} kVAR = Q ✓ ;  V·I·cos φ = {_c3(V*I*FP/1000)} kW = P ✓")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 3 ---")
quiz_numerico("Corriente que toma la trilladora (P = 18,0 kW, 240 V, FP 0,680) (en A):",
    110, 0.03, "A", "S=18,0/0,680=26,5 kVA; I=26,5×10³/240=110 A.", id_preg="CA_p3_1", peso=3.0)
quiz_numerico("Potencia reactiva Q de la trilladora (en kVAR):",
    19.4, 0.03, "kVAR", "Q=P·tan(arccos 0,680)=18,0·1,08=19,4 kVAR.", id_preg="CA_p3_2", peso=2.0)
quiz_opcion_multiple("Con Q/P = 1,08, ¿qué dice la regla de la CREG sobre la energía reactiva de la trilladora?",
    ['Se cobra la reactiva que supere el 50 % de la activa: aquí hay exceso facturable',
     'No se cobra nada: la energía reactiva no hace trabajo y por eso es gratuita',
     'Se cobra toda la energía reactiva, sin importar cuánta energía activa se consuma',
     'Solo se cobra si el factor de potencia es capacitivo, no inductivo como aquí'], 0,
    "Res. CREG 015 de 2018 (art. 16, cap. 12) y Res. CREG 101 035 de 2024: se cobra la reactiva inductiva que supere el 50 % de la activa en cada periodo horario (FP inductivo < 0,90).",
    id_preg="CA_p3_3", peso=2.0)
quiz_opcion_multiple("DEFIENDE TU CONCLUSIÓN: el dueño de la trilladora dice que 'la potencia reactiva no se consume, va y vuelve, así que tener un factor de potencia bajo no le cuesta nada a nadie'. ¿Correcto o solo convincente?",
    ['Solo convincente: la reactiva no hace trabajo, pero sí aumenta la corriente; eso eleva las pérdidas I²R y se cobra el exceso',
     'Correcto: la potencia media de una reactancia pura es cero, así que la reactiva no tiene ningún efecto real',
     'Correcto: el medidor de energía solo registra kWh, de modo que la reactiva nunca aparece en la factura',
     'Solo convincente: la reactiva sí se consume como calor en los motores, igual que la potencia activa'], 0,
    'Es cierto que la reactiva no se convierte en trabajo, y eso hace convincente la idea. Pero con el mismo P, la corriente sube de 75,0 A a 110 A y las pérdidas en el cable se duplican. La opción «se consume como calor en los motores» acierta el veredicto con una razón falsa.',
    id_preg='CA_p3_concep_3', peso=2.0)

def _interact_coma(fn, _crudos=(), **controles):
    # Como interact, con lectura de sliders en coma decimal. La salida NO se captura con «with salida:»:
    # se calculan el texto y las figuras y se escriben con append_stdout/append_display_data
    # (evita bloqueos de Voilà en Binder y funciona también en JupyterLite).
    import ipywidgets as _w, io as _io, contextlib as _cl
    import matplotlib.pyplot as _plt
    def _fmt(v, paso, crudo=False):
        # Lectura con coma decimal y 3 cifras significativas (×10ⁿ fuera de 0,001 ≤ |v| < 1000).
        # IntSlider: entero exacto. «crudo»: decimales del paso (slider que representa una medición sin redondear).
        import math
        if isinstance(v, int) and not isinstance(v, bool): return str(v)
        if crudo:
            s = repr(float(paso)); dec = 0 if 'e' in s else len(s.split('.')[1].rstrip('0'))
            return f"{v:.{min(dec, 4)}f}".replace('.', ',').replace('-', '−')
        v = float(v)
        if v == 0 or not math.isfinite(v): return '0' if v == 0 else str(v)
        v = float(f"{v:.2e}")
        if abs(v) >= 1000 or abs(v) < 0.001:
            m, e = f"{v:.2e}".split('e')
            return m.replace('.', ',').replace('-', '−') + '×10' + str(int(e)).translate(str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹'))
        t = f"{v:#.3g}"
        return (t[:-1] if t.endswith('.') else t).replace('.', ',').replace('-', '−')
    filas = []
    for nombre, c in controles.items():
        if isinstance(c, (_w.FloatSlider, _w.IntSlider)):
            c.readout = False
            lab = _w.Label(layout=_w.Layout(min_width='90px'))
            def _act(*_, c=c, lab=lab, cr=(nombre in _crudos)): lab.value = _fmt(c.value, c.step, cr)
            c.observe(_act, 'value'); _act()
            filas.append(_w.HBox([c, lab]))
        else:
            filas.append(c)
    salida = _w.Output()
    def _ejecutar(*_):
        partes, buf = [], _io.StringIO()
        def _vaciar_texto():
            t = buf.getvalue()
            if t: partes.append(('t', t)); buf.seek(0); buf.truncate(0)
        def _capturar(*a, **k):
            _vaciar_texto()
            for num in _plt.get_fignums():
                b = _io.BytesIO(); _plt.figure(num).savefig(b, format='png', bbox_inches='tight')
                partes.append(('i', b.getvalue()))
            _plt.close('all')
        _show = _plt.show; _plt.show = _capturar
        try:
            with _cl.redirect_stdout(buf):
                try: fn(**{k: c.value for k, c in controles.items()})
                except Exception as e: print('⚠ No se pudo calcular con estos valores:', e)
            _capturar()
        finally:
            _plt.show = _show
        _vaciar_texto()
        # Reemplazo en UNA sola asignación: la salida nunca queda vacía (sin saltos de la página)
        import base64 as _b64
        nuevas = []
        for tipo, val in partes:
            if tipo == 't': nuevas.append({'output_type': 'stream', 'name': 'stdout', 'text': val})
            else: nuevas.append({'output_type': 'display_data', 'metadata': {},
                                 'data': {'image/png': _b64.b64encode(val).decode('ascii'), 'text/plain': '<Figure>'}})
        salida.outputs = tuple(nuevas)
    for c in controles.values():
        c.observe(_ejecutar, 'value')
    _ejecutar()
    display(_w.VBox(filas + [salida]))

def _explora_sin_cifras(fn):
    # Opción 3: exploración cualitativa; las cifras de este punto solo están en «Mostrar solución».
    import functools, io, contextlib, re as _re
    _pat = _re.compile(r'(?<![\w.,/])[-−]?\d+(?:[.,]\d+)?(?:\s?×\s?10[⁻⁰¹²³⁴⁵⁶⁷⁸⁹]+|[eE][+-]?\d+)?(?![\wπ·(]|[.,]\d)')
    def _m(t): return _pat.sub('…', t)
    def _ocultar():
        for num in plt.get_fignums():
            fig = plt.figure(num)
            textos = list(fig.texts) + ([fig._suptitle] if getattr(fig, '_suptitle', None) else [])
            for ax in fig.axes:
                textos += [ax.title, ax._left_title, ax._right_title] + list(ax.texts)
                if ax.get_legend(): textos += list(ax.get_legend().get_texts())
            for t in textos: t.set_text(_m(t.get_text()))
    @functools.wraps(fn)
    def envuelta(*a, **k):
        _show = plt.show
        def _show2(*sa, **sk):
            _ocultar(); _show(*sa, **sk)
        buf = io.StringIO(); plt.show = _show2
        try:
            with contextlib.redirect_stdout(buf): fn(*a, **k)
            _ocultar()
        finally:
            plt.show = _show
        for linea in buf.getvalue().splitlines():
            print(linea if ('⚠' in linea or '⛔' in linea) else _m(linea))
        print("(Exploración cualitativa: observa tendencias. Las cifras de este punto están en «Mostrar solución».)")
    return envuelta

print("\n--- 🔬 Explora después de responder ---")
print("Los controles NO arrancan en los datos del enunciado. Primero resuelve y responde;\nluego explora tendencias, casos límite y comprueba tu razonamiento.")
_interact_coma(_explora_sin_cifras(trilladora),
    P_kW=FloatSlider(value=10.0,min=2,max=40,step=1,description='P (kW)'),
    FP=FloatSlider(value=0.8,min=0.5,max=1.0,step=0.02,description='FP'),
    V=FloatSlider(value=208.0,min=120,max=480,step=8,description='V (V)'),
    R_linea=FloatSlider(value=0.1,min=0.01,max=0.2,step=0.01,description='R línea (Ω)'));

---
## Punto 4 — Banco de condensadores: corrección del factor de potencia (Ing. Eléctrica/Eficiencia energética)  ★★★  [PRC, ULC]
Para dejar de pagar exceso de reactiva, la trilladora del Punto 3 (P = 18,0 kW, FP = 0,680, 240 V, 60,0 Hz)
instala un **banco de condensadores en paralelo** con la carga, para llevar el FP a **0,950 en atraso**. Los motores
siguen haciendo el mismo trabajo: P no cambia.

**Reactiva que debe aportar el banco:**  Q_C = Q₁ − Q₂ = P·(tan φ₁ − tan φ₂)

**Capacitancia:**  C = Q_C/(V²·ω)

**Corriente nueva:**  I₂ = P/(V·FP₂)

> 🔧 **Aplicación en ingeniería:** un condensador toma corriente adelantada 90°, justo al contrario de la bobina; en
> paralelo con los motores, aporta localmente la reactiva que antes viajaba desde la red. Corregir de más tampoco es
> gratis: si el banco aporta más reactiva de la que necesita la carga, el FP se vuelve **capacitivo** y la corriente
> vuelve a subir. La Res. CREG 101 035 de 2024 también limita la reactiva capacitiva: en los niveles de tensión 1 y 2,
> como máximo el 50 % de la activa en el periodo de facturación (FP capacitivo ≥ 0,90). Un banco fijo grande es el riesgo
> típico: cuando los motores se apagan, el banco queda conectado casi solo.

> 🤖 **TIP-IA — Generador de problemas:**
> Pídele a un asistente: *"Créame una variante de la corrección del factor de potencia con otra carga, otro FP inicial y otro FP objetivo, y pídeme calcular Q₁, Q₂, la capacitancia C = Q_C/(V²ω) y la nueva corriente. No me des la solución hasta que yo la intente."* Primero resuelves tú, después verificas.


In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as _tk_br
if not getattr(_tk_br.ScalarFormatter, '_coma_br', False):
    # coma decimal en las marcas de los ejes y en el texto de desplazamiento (offset)
    _fmt_br_call, _fmt_br_off = _tk_br.ScalarFormatter.__call__, _tk_br.ScalarFormatter.get_offset
    _tk_br.ScalarFormatter.__call__ = lambda self, x, pos=None: _fmt_br_call(self, x, pos).replace('.', ',')
    def _off_br(self):
        import re as _re_br
        sup = str.maketrans('−-+0123456789', '⁻⁻⁺⁰¹²³⁴⁵⁶⁷⁸⁹')
        t = _fmt_br_off(self).replace('.', ',')
        t = _re_br.sub(r'^1e([−+-]?\d+)', lambda m: '×10' + m.group(1).translate(sup), t)
        return _re_br.sub(r'(\d)e([−+-]?\d+)', lambda m: m.group(1) + '×10' + m.group(2).translate(sup), t)
    _tk_br.ScalarFormatter.get_offset = _off_br
    _tk_br.ScalarFormatter._coma_br = True
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
# --- pantalla angosta (celular) ---
try:
    from IPython.display import HTML as _HTML_br
    display(_HTML_br('<style>'
        '.widget-radio{width:100% !important;max-width:100%;height:auto !important}'
        '.widget-radio-box{width:100%;max-width:100%}'
        '.widget-radio-box label{display:flex !important;flex-direction:row-reverse;justify-content:flex-end;align-items:flex-start;gap:8px;'
        'white-space:normal !important;height:auto !important;line-height:1.35 !important;margin:3px 0 7px 0;overflow-wrap:anywhere}'
        '.widget-radio-box input{flex:0 0 auto;margin:3px 0 0 0}'
        '.jp-RenderedText pre,.widget-output pre{word-break:normal !important;overflow-wrap:anywhere;white-space:pre-wrap !important}'
        '.widget-html-content{white-space:normal;line-height:1.4}'
        '.widget-text{max-width:100%}'
        '</style>'))
except Exception:
    pass
if not getattr(plt, '_dos_paneles_br', False):
    def _show_br(*a, _orig=plt.show, **k):
        # Figura de dos paneles lado a lado: se entrega como dos imágenes que se apilan si la pantalla es angosta.
        # El PNG completo viaja en el mismo paquete (para Colab sin HTML, miniaturas y pruebas).
        import io as _io, base64 as _b64
        from matplotlib.transforms import Bbox as _Bbox
        for _num in list(plt.get_fignums()):
            fig = plt.figure(_num)
            try:
                grupos = {}
                for ax in fig.axes: grupos.setdefault(round(ax.get_position().x0, 2), []).append(ax)
                if len(grupos) != 2 or len({round(ax.get_position().y0, 2) for ax in fig.axes}) != 1: continue
                fig.canvas.draw(); r = fig.canvas.get_renderer()
                def _png(bb='tight', dpi=None):
                    b = _io.BytesIO(); fig.savefig(b, format='png', bbox_inches=bb, dpi=dpi); return _b64.b64encode(b.getvalue()).decode('ascii')
                partes = []
                for x0 in sorted(grupos):
                    bb = _Bbox.union([ax.get_tightbbox(r) for ax in grupos[x0]]).transformed(fig.dpi_scale_trans.inverted())
                    partes.append(_png(bb.padded(0.06), 144))
                html = ('<div style="display:flex;flex-wrap:wrap;gap:8px 16px;align-items:flex-start">'
                        + ''.join(f'<img alt="Esquema del punto (panel {i + 1} de 2)" src="data:image/png;base64,{p}" '
                                  'style="flex:1 1 300px;min-width:0;max-width:min(100%,420px);height:auto">' for i, p in enumerate(partes)) + '</div>')
                display({'text/html': html, 'image/png': _png(), 'text/plain': '<Figure>'}, raw=True)
                plt.close(fig)
            except Exception:
                pass
        return _orig(*a, **k)
    plt.show = _show_br; plt._dos_paneles_br = True
# --- fin pantalla angosta ---
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))


def _num_coma(s):
    import re as _re
    # reemplaza punto decimal por coma en números dentro de un string ya formateado
    return _re.sub(r"(\d)\.(\d)", r"\1,\2", s)
def print_es(*args, **kwargs):
    args2=[_num_coma(a) if isinstance(a,str) else a for a in args]
    print(*args2, **kwargs)

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

try:
    print_es
except NameError:
    def print_es(*args, **kwargs):
        import re as _re
        print(*[_re.sub(r"(\d)\.(\d)", r"\1,\2", a) if isinstance(a, str) else a for a in args], **kwargs)
# Signo menos tipográfico (−) en todo texto impreso: «-5» → «−5» (no toca guiones entre palabras, rangos ni 1e-3)
import builtins as _bi_br
if not getattr(_bi_br.print, '_menos_br', False):
    def _print_br(*a, _orig=_bi_br.print, **k):
        import re as _re
        _orig(*[_re.sub(r'(?<![\w.,\-])-(?=\d)', '−', x) if isinstance(x, str) else x for x in a], **k)
    _print_br._menos_br = True
    _bi_br.print = _print_br
# --- Cuestionarios Física BR v3: opciones barajadas, crédito por intento, nota sobre el total ---
import random as _rnd_br
try:
    PESOS_BR
except NameError:
    PESOS_BR = {}      # id_preg -> peso de cada pregunta mostrada
try:
    INTENTOS_BR
except NameError:
    INTENTOS_BR = {}   # id_preg -> [intentos, resuelta]
_CREDITO_BR = {1: 1.0, 2: 0.5, 3: 0.25}   # crédito según el intento en que se acierta
_MAX_INT_BR = 3

def _evaluar_intento(id_preg, ok, peso, explicacion, msg_ok, msg_mal):
    est = INTENTOS_BR.setdefault(id_preg, [0, False])
    if est[1]:
        print("✔ Ya resolviste esta pregunta; tu crédito quedó registrado.")
        if explicacion: print_es("→ " + explicacion)
        return
    if est[0] >= _MAX_INT_BR:
        print("⛔ Agotaste los 3 intentos con crédito: esta pregunta ya no suma.")
        if explicacion: print_es("→ " + explicacion)
        return
    est[0] += 1
    n = est[0]
    if ok:
        est[1] = True
        cred = _CREDITO_BR.get(n, 0.0)
        REGISTRO_BR[id_preg] = (cred, float(peso))
        print_es(msg_ok + f"  (intento {n}: crédito {int(cred*100)} %)")
        if explicacion: print_es("→ " + explicacion)
    else:
        REGISTRO_BR.setdefault(id_preg, (0.0, float(peso)))
        if n < _MAX_INT_BR:
            print(msg_mal + f"  Intento {n} de {_MAX_INT_BR}. Revisa tu razonamiento antes de volver a intentar.")
        else:
            print(msg_mal + " Agotaste los 3 intentos: esta pregunta ya no suma. Estudia la explicación:")
            if explicacion: print_es("→ " + explicacion)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    orden = list(range(len(opciones))); _rnd_br.shuffle(orden)   # orden distinto en cada sesión
    correcta = orden.index(indice_correcto)
    letras = "abcdefgh"
    etiquetas = [f"{letras[i]}) {opciones[j]}" for i, j in enumerate(orden)]
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=etiquetas, value=None, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            if radio.index is None:
                print("⚠ Elige una opción."); return
            _evaluar_intento(id_preg, radio.index == correcta, peso, explicacion,
                             "✅ ¡Correcto!", "❌ Incorrecto.")
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.strip().replace(",", "."))
            except ValueError: print("⚠ Escribe un número (usa coma o punto decimal)."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            _evaluar_intento(id_preg, ok, peso, explicacion,
                             f"✅ ¡Correcto!  ({valor_correcto:.3g} {unidad})", "❌ Fuera de rango.")
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not PESOS_BR:
                print("No se encontraron cuestionarios en este taller."); return
            peso_total = sum(PESOS_BR.values())
            peso_ok = sum(float(c)*p for c, p in REGISTRO_BR.values())
            nota = 5.0*peso_ok/peso_total if peso_total > 0 else 0.0
            total = len(PESOS_BR)
            resp = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0])[0] > 0)
            prim = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0, False])[1] and INTENTOS_BR[k][0] == 1)
            print("="*46)
            print("          CALIFICACIÓN FORMATIVA")
            print("="*46)
            print(f"  Preguntas respondidas:       {resp} de {total}")
            print(f"  Acertadas al primer intento: {prim}")
            print_es(f"  Puntaje ponderado:           {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*46)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*46)
            if resp < total: print(f"  ⚠ Faltan {total-resp} preguntas: cuentan como 0.")
            if nota >= 3.5: print("  ✅ ¡Buen trabajo! Dominas los conceptos.")
            elif nota >= 3.0: print("  🟡 Vas bien, repasa lo que fallaste.")
            else: print("  🔴 Repasa la teoría y vuelve a los puntos.")
            print("\n  Crédito por pregunta: 100 % al primer intento,")
            print("  50 % al segundo, 25 % al tercero. Predice y")
            print("  razona ANTES de verificar: adivinar cuesta nota.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>📊 Tu calificación del taller</h3>"
        "<i>La nota se calcula sobre TODAS las preguntas del taller y premia acertar al primer intento.</i>"),
        boton, salida]))


try:
    import matplotlib.pyplot as _plt_style
    _plt_style.rcParams['axes.spines.top']=False
    _plt_style.rcParams['axes.spines.right']=False
    _plt_style.rcParams['axes.titlesize']=12
    _plt_style.rcParams['axes.titleweight']='bold'
    _plt_style.rcParams['axes.titlecolor']='#2c3e50'
    _plt_style.rcParams['axes.labelsize']=10
    _plt_style.rcParams['axes.labelcolor']='#34495e'
    _plt_style.rcParams['font.size']=10
    _plt_style.rcParams['axes.grid']=True
    _plt_style.rcParams['grid.alpha']=0.25
    _plt_style.rcParams['grid.linestyle']='--'
    _plt_style.rcParams['axes.edgecolor']='#7f8c8d'
    _plt_style.rcParams['xtick.color']='#34495e'
    _plt_style.rcParams['ytick.color']='#34495e'
    _plt_style.rcParams['figure.facecolor']='white'
    _plt_style.rcParams['axes.facecolor']='#fcfcfd'
except Exception:
    pass
# --- fin estilo ---


def _c3(x):
    # 3 cifras significativas conservando ceros finales (8,00 y no 8); notación ×10ⁿ desde 1000
    if x != 0 and abs(x) >= 1000: return _sci(x)
    t = f"{x:#.3g}"
    return t[:-1] if t.endswith('.') else t

def _sci(x):
    e = int(np.floor(np.log10(abs(x)))); m = x/10**e
    if round(m, 2) >= 10: m /= 10; e += 1
    return f"{m:.2f}×10" + str(e).translate(str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹'))

def _3c(x):
    # 3 cifras significativas con coma decimal; ×10ⁿ fuera de 0,001 ≤ |x| < 1000 (textos de gráficas)
    import math
    try: x = float(x)
    except (TypeError, ValueError): return str(x)
    if x == 0 or not math.isfinite(x): return str(x).replace('.', ',') if x else '0'
    x = float(f"{x:.2e}")
    if abs(x) >= 1000 or abs(x) < 0.001:
        e = int(math.floor(math.log10(abs(x)))); m = x / 10**e
        if round(abs(m), 2) >= 10: m /= 10; e += 1
        return f"{m:.2f}".replace('.', ',').replace('-', '−') + '×10' + str(e).translate(str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹'))
    t = f"{x:#.3g}"
    return (t[:-1] if t.endswith('.') else t).replace('.', ',').replace('-', '−')

def esquema():
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.8))
    P = 18.0; Q1 = P*np.tan(np.arccos(0.68)); Q2 = P*np.tan(np.arccos(0.95))
    a1.annotate('', xy=(P, 0), xytext=(0, 0), arrowprops=dict(arrowstyle='-|>', color='#27ae60', lw=2.5))
    a1.annotate('', xy=(P, Q1), xytext=(0, 0), arrowprops=dict(arrowstyle='-|>', color='#8e44ad', lw=2))
    a1.annotate('', xy=(P, Q2), xytext=(0, 0), arrowprops=dict(arrowstyle='-|>', color='#2980b9', lw=2.5))
    a1.annotate('', xy=(P + 0.8, Q2), xytext=(P + 0.8, Q1), arrowprops=dict(arrowstyle='-|>', color='#c0392b', lw=2.5))
    a1.text(P + 1.3, (Q1 + Q2)/2, 'Q_C del\nbanco', fontsize=8, color='#c0392b')
    a1.text(4, Q1*0.62, 'S₁ (antes)', fontsize=8, color='#8e44ad'); a1.text(10, 1.5, 'S₂ (después)', fontsize=8, color='#2980b9')
    a1.set_xlim(-1, 26); a1.set_ylim(-1, 22); a1.set_aspect('equal'); a1.axis('off'); a1.set_title('El banco resta reactiva: mismo P, menor S', fontsize=10)
    Cs = np.linspace(0, 1500, 400)*1e-6; w = 2*np.pi*60; Qn = Q1*1000 - 240**2*w*Cs
    a2.plot(Cs*1e6, np.hypot(18000, Qn)/240, color='#8e44ad', lw=2)
    a2.set_xlabel('capacitancia del banco (µF)'); a2.set_ylabel('corriente de la red (A)')
    a2.set_title('Hay un mínimo: corregir de más vuelve a subir la corriente', fontsize=10)
    plt.tight_layout(); plt.show()
esquema()

def banco(C_uF=400.0, P_kW=18.0, FP1=0.68, V=240.0):
    w = 2*np.pi*60; P = P_kW*1000; Q1 = P*np.tan(np.arccos(FP1)); Qc = V**2*w*C_uF*1e-6; Q = Q1 - Qc
    S = np.hypot(P, Q); FP = P/S; I = S/V
    tipo = 'en atraso (inductivo)' if Q >= 0 else 'en adelanto (capacitivo)'
    print_es(f"Q_C = {_c3(Qc/1000)} kVAR   |   Q neta = {_c3(Q/1000)} kVAR   |   FP = {_c3(FP)} {tipo}   |   I = {_c3(I)} A")
    if Q < 0: print_es("⚠ Sobrecorrección: el banco inyecta reactiva capacitiva a la red." + (" ⛔ Supera el 50 % de la activa: límite capacitivo excedido." if -Q/P > 0.5 else ""))
    elif Q/P > 0.5: print_es("⚠ Todavía hay exceso de reactiva inductiva (Q/P > 0,500).")
    Cs = np.linspace(0, 2*max(C_uF, Q1/(V**2*w)*1e6), 400)
    plt.figure(figsize=(8, 3.3)); plt.plot(Cs, np.hypot(P, Q1 - V**2*w*Cs*1e-6)/V, color='#8e44ad', lw=2); plt.scatter([C_uF], [I], color='#c0392b', zorder=5)
    plt.xlabel('C (µF)'); plt.ylabel('I (A)'); plt.title('Corriente de la red según el banco instalado', fontsize=10); plt.tight_layout(); plt.show()

def solucion():
    P = 18000.0; V = 240.0; w = 2*np.pi*60; Q1 = P*np.tan(np.arccos(0.68)); Q2 = P*np.tan(np.arccos(0.95)); Qc = Q1 - Q2; C = Qc/(V**2*w)
    print_es(f"a) Q₁ = {_c3(Q1/1000)} kVAR (Punto 3) ;  tan φ₂ = tan(arccos 0,950) = {_c3(np.tan(np.arccos(0.95)))} ;  Q₂ = 18,0·{_c3(np.tan(np.arccos(0.95)))} = {_c3(Q2/1000)} kVAR")
    print_es(f"b) Q_C = Q₁ − Q₂ = {_c3(Q1/1000)} − {_c3(Q2/1000)} = {_c3(Qc/1000)} kVAR")
    print_es(f"c) C = Q_C/(V²·ω) = {_c3(Qc)}/(240²·{_c3(w)}) = {_sci(C)} F = {_c3(C*1e6)} µF")
    I2 = P/(V*0.95)
    print_es(f"d) I₂ = P/(V·FP₂) = 18,0×10³/(240·0,950) = {_c3(I2)} A  (antes 110 A) ;  pérdidas: {_c3(I2**2*0.05)} W (antes 608 W)")
    print_es(f"e) Q₂/P = {_c3(Q2/P)} < 0,500: ya no hay exceso de reactiva inductiva.")
    Cd = 2*C; Qd = Q1 - V**2*w*Cd; Sd = np.hypot(P, Qd)
    print_es(f"f) Con el doble de capacitancia ({_c3(Cd*1e3)} mF): Q = −{_c3(-Qd/1000)} kVAR (capacitiva), FP = {_c3(P/Sd)} en adelanto, I = {_c3(Sd/V)} A: peor que con C.")
    print_es(f"   |Q|/P = {_c3(-Qd/P)} < 0,500: aún dentro del límite capacitivo de nivel 1, pero se pagó el doble de banco para tener más corriente;")
    print_es(f"   y si los motores se apagan, el banco queda solo inyectando {_c3(V**2*w*Cd/1000)} kVAR capacitivos.")
    St = complex(P, Q1) + complex(0, -V**2*w*C)
    print_es(f"Verificación con potencias complejas: S = P + jQ₁ − jV²ωC → FP = {_c3(St.real/abs(St))} ✓ , I = {_c3(abs(St)/V)} A ✓")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 4 ---")
quiz_numerico("Potencia reactiva Q_C que debe aportar el banco para pasar de FP 0,680 a 0,950 (en kVAR):",
    13.5, 0.03, "kVAR", "Q_C=Q₁−Q₂=19,4−5,92=13,5 kVAR.", id_preg="CA_p4_1", peso=2.0)
quiz_numerico("Capacitancia del banco a 240 V y 60,0 Hz, C = Q_C/(V²·ω) (en µF):",
    621, 0.03, "µF", "C=13,5×10³/(240²·377)=6,21×10⁻⁴ F=621 µF.", id_preg="CA_p4_2", peso=3.0)
quiz_numerico("Corriente que toma la trilladora después de la corrección (en A):",
    78.9, 0.03, "A", "I₂=18,0×10³/(240·0,950)=78,9 A (antes 110 A).", id_preg="CA_p4_3", peso=3.0)
quiz_opcion_multiple("DEFIENDE TU CONCLUSIÓN: un instalador propone poner el doble de capacitancia (1,24 mF), 'porque entre más condensadores, mejor factor de potencia y menos corriente'. ¿Correcto o solo convincente?",
    ['Solo convincente: con el doble, el FP pasa a capacitivo (0,922) y la corriente sube de 78,9 A a 81,4 A: más banco, peor resultado',
     'Correcto: cada condensador adicional reduce la corriente, así que el máximo posible es siempre la mejor opción',
     'Correcto: el factor de potencia nunca puede pasar de 1, de modo que añadir más condensadores no tiene riesgo',
     'Solo convincente: los condensadores consumen mucha potencia activa y por eso aumentan la factura de energía'], 0,
    'La corriente tiene un mínimo cuando Q_C = Q₁ (FP = 1, con 894 µF). Más allá, la reactiva capacitiva crece y la corriente vuelve a subir. Con 1,24 mF todavía no se excede el límite capacitivo (|Q|/P = 0,421 < 0,500), pero el banco queda sobredimensionado y, con los motores apagados, inyecta reactiva a la red. La opción «consumen mucha potencia activa» acierta el veredicto con una razón falsa: un condensador ideal no consume potencia activa.',
    id_preg='CA_p4_concep_4', peso=2.0)

def _interact_coma(fn, _crudos=(), **controles):
    # Como interact, con lectura de sliders en coma decimal. La salida NO se captura con «with salida:»:
    # se calculan el texto y las figuras y se escriben con append_stdout/append_display_data
    # (evita bloqueos de Voilà en Binder y funciona también en JupyterLite).
    import ipywidgets as _w, io as _io, contextlib as _cl
    import matplotlib.pyplot as _plt
    def _fmt(v, paso, crudo=False):
        # Lectura con coma decimal y 3 cifras significativas (×10ⁿ fuera de 0,001 ≤ |v| < 1000).
        # IntSlider: entero exacto. «crudo»: decimales del paso (slider que representa una medición sin redondear).
        import math
        if isinstance(v, int) and not isinstance(v, bool): return str(v)
        if crudo:
            s = repr(float(paso)); dec = 0 if 'e' in s else len(s.split('.')[1].rstrip('0'))
            return f"{v:.{min(dec, 4)}f}".replace('.', ',').replace('-', '−')
        v = float(v)
        if v == 0 or not math.isfinite(v): return '0' if v == 0 else str(v)
        v = float(f"{v:.2e}")
        if abs(v) >= 1000 or abs(v) < 0.001:
            m, e = f"{v:.2e}".split('e')
            return m.replace('.', ',').replace('-', '−') + '×10' + str(int(e)).translate(str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹'))
        t = f"{v:#.3g}"
        return (t[:-1] if t.endswith('.') else t).replace('.', ',').replace('-', '−')
    filas = []
    for nombre, c in controles.items():
        if isinstance(c, (_w.FloatSlider, _w.IntSlider)):
            c.readout = False
            lab = _w.Label(layout=_w.Layout(min_width='90px'))
            def _act(*_, c=c, lab=lab, cr=(nombre in _crudos)): lab.value = _fmt(c.value, c.step, cr)
            c.observe(_act, 'value'); _act()
            filas.append(_w.HBox([c, lab]))
        else:
            filas.append(c)
    salida = _w.Output()
    def _ejecutar(*_):
        partes, buf = [], _io.StringIO()
        def _vaciar_texto():
            t = buf.getvalue()
            if t: partes.append(('t', t)); buf.seek(0); buf.truncate(0)
        def _capturar(*a, **k):
            _vaciar_texto()
            for num in _plt.get_fignums():
                b = _io.BytesIO(); _plt.figure(num).savefig(b, format='png', bbox_inches='tight')
                partes.append(('i', b.getvalue()))
            _plt.close('all')
        _show = _plt.show; _plt.show = _capturar
        try:
            with _cl.redirect_stdout(buf):
                try: fn(**{k: c.value for k, c in controles.items()})
                except Exception as e: print('⚠ No se pudo calcular con estos valores:', e)
            _capturar()
        finally:
            _plt.show = _show
        _vaciar_texto()
        # Reemplazo en UNA sola asignación: la salida nunca queda vacía (sin saltos de la página)
        import base64 as _b64
        nuevas = []
        for tipo, val in partes:
            if tipo == 't': nuevas.append({'output_type': 'stream', 'name': 'stdout', 'text': val})
            else: nuevas.append({'output_type': 'display_data', 'metadata': {},
                                 'data': {'image/png': _b64.b64encode(val).decode('ascii'), 'text/plain': '<Figure>'}})
        salida.outputs = tuple(nuevas)
    for c in controles.values():
        c.observe(_ejecutar, 'value')
    _ejecutar()
    display(_w.VBox(filas + [salida]))

def _explora_sin_cifras(fn):
    # Opción 3: exploración cualitativa; las cifras de este punto solo están en «Mostrar solución».
    import functools, io, contextlib, re as _re
    _pat = _re.compile(r'(?<![\w.,/])[-−]?\d+(?:[.,]\d+)?(?:\s?×\s?10[⁻⁰¹²³⁴⁵⁶⁷⁸⁹]+|[eE][+-]?\d+)?(?![\wπ·(]|[.,]\d)')
    def _m(t): return _pat.sub('…', t)
    def _ocultar():
        for num in plt.get_fignums():
            fig = plt.figure(num)
            textos = list(fig.texts) + ([fig._suptitle] if getattr(fig, '_suptitle', None) else [])
            for ax in fig.axes:
                textos += [ax.title, ax._left_title, ax._right_title] + list(ax.texts)
                if ax.get_legend(): textos += list(ax.get_legend().get_texts())
            for t in textos: t.set_text(_m(t.get_text()))
    @functools.wraps(fn)
    def envuelta(*a, **k):
        _show = plt.show
        def _show2(*sa, **sk):
            _ocultar(); _show(*sa, **sk)
        buf = io.StringIO(); plt.show = _show2
        try:
            with contextlib.redirect_stdout(buf): fn(*a, **k)
            _ocultar()
        finally:
            plt.show = _show
        for linea in buf.getvalue().splitlines():
            print(linea if ('⚠' in linea or '⛔' in linea) else _m(linea))
        print("(Exploración cualitativa: observa tendencias. Las cifras de este punto están en «Mostrar solución».)")
    return envuelta

print("\n--- 🔬 Explora después de responder ---")
print("Los controles NO arrancan en los datos del enunciado. Primero resuelve y responde;\nluego explora tendencias, casos límite y comprueba tu razonamiento.")
_interact_coma(_explora_sin_cifras(banco),
    C_uF=FloatSlider(value=300.0,min=0,max=1500,step=25,description='C (µF)'),
    P_kW=FloatSlider(value=25.0,min=5,max=40,step=1,description='P (kW)'),
    FP1=FloatSlider(value=0.74,min=0.5,max=0.98,step=0.02,description='FP inicial'),
    V=FloatSlider(value=208.0,min=120,max=480,step=8,description='V (V)'));

---
## ✅ Fin del taller complementario de Corriente Alterna (Física BR)
Del tomacorriente a la factura: valor eficaz, reactancia, impedancia y la corrección del factor de potencia en una planta.
Sigue con el taller de **Ecuaciones de Maxwell**. Experimenta con los sliders y **predice** antes de pulsar "Mostrar solución".
*Física interactiva para ingeniería — 2026-2.*


In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as _tk_br
if not getattr(_tk_br.ScalarFormatter, '_coma_br', False):
    # coma decimal en las marcas de los ejes y en el texto de desplazamiento (offset)
    _fmt_br_call, _fmt_br_off = _tk_br.ScalarFormatter.__call__, _tk_br.ScalarFormatter.get_offset
    _tk_br.ScalarFormatter.__call__ = lambda self, x, pos=None: _fmt_br_call(self, x, pos).replace('.', ',')
    def _off_br(self):
        import re as _re_br
        sup = str.maketrans('−-+0123456789', '⁻⁻⁺⁰¹²³⁴⁵⁶⁷⁸⁹')
        t = _fmt_br_off(self).replace('.', ',')
        t = _re_br.sub(r'^1e([−+-]?\d+)', lambda m: '×10' + m.group(1).translate(sup), t)
        return _re_br.sub(r'(\d)e([−+-]?\d+)', lambda m: m.group(1) + '×10' + m.group(2).translate(sup), t)
    _tk_br.ScalarFormatter.get_offset = _off_br
    _tk_br.ScalarFormatter._coma_br = True
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
# --- pantalla angosta (celular) ---
try:
    from IPython.display import HTML as _HTML_br
    display(_HTML_br('<style>'
        '.widget-radio{width:100% !important;max-width:100%;height:auto !important}'
        '.widget-radio-box{width:100%;max-width:100%}'
        '.widget-radio-box label{display:flex !important;flex-direction:row-reverse;justify-content:flex-end;align-items:flex-start;gap:8px;'
        'white-space:normal !important;height:auto !important;line-height:1.35 !important;margin:3px 0 7px 0;overflow-wrap:anywhere}'
        '.widget-radio-box input{flex:0 0 auto;margin:3px 0 0 0}'
        '.jp-RenderedText pre,.widget-output pre{word-break:normal !important;overflow-wrap:anywhere;white-space:pre-wrap !important}'
        '.widget-html-content{white-space:normal;line-height:1.4}'
        '.widget-text{max-width:100%}'
        '</style>'))
except Exception:
    pass
if not getattr(plt, '_dos_paneles_br', False):
    def _show_br(*a, _orig=plt.show, **k):
        # Figura de dos paneles lado a lado: se entrega como dos imágenes que se apilan si la pantalla es angosta.
        # El PNG completo viaja en el mismo paquete (para Colab sin HTML, miniaturas y pruebas).
        import io as _io, base64 as _b64
        from matplotlib.transforms import Bbox as _Bbox
        for _num in list(plt.get_fignums()):
            fig = plt.figure(_num)
            try:
                grupos = {}
                for ax in fig.axes: grupos.setdefault(round(ax.get_position().x0, 2), []).append(ax)
                if len(grupos) != 2 or len({round(ax.get_position().y0, 2) for ax in fig.axes}) != 1: continue
                fig.canvas.draw(); r = fig.canvas.get_renderer()
                def _png(bb='tight', dpi=None):
                    b = _io.BytesIO(); fig.savefig(b, format='png', bbox_inches=bb, dpi=dpi); return _b64.b64encode(b.getvalue()).decode('ascii')
                partes = []
                for x0 in sorted(grupos):
                    bb = _Bbox.union([ax.get_tightbbox(r) for ax in grupos[x0]]).transformed(fig.dpi_scale_trans.inverted())
                    partes.append(_png(bb.padded(0.06), 144))
                html = ('<div style="display:flex;flex-wrap:wrap;gap:8px 16px;align-items:flex-start">'
                        + ''.join(f'<img alt="Esquema del punto (panel {i + 1} de 2)" src="data:image/png;base64,{p}" '
                                  'style="flex:1 1 300px;min-width:0;max-width:min(100%,420px);height:auto">' for i, p in enumerate(partes)) + '</div>')
                display({'text/html': html, 'image/png': _png(), 'text/plain': '<Figure>'}, raw=True)
                plt.close(fig)
            except Exception:
                pass
        return _orig(*a, **k)
    plt.show = _show_br; plt._dos_paneles_br = True
# --- fin pantalla angosta ---
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))


def _num_coma(s):
    import re as _re
    # reemplaza punto decimal por coma en números dentro de un string ya formateado
    return _re.sub(r"(\d)\.(\d)", r"\1,\2", s)
def print_es(*args, **kwargs):
    args2=[_num_coma(a) if isinstance(a,str) else a for a in args]
    print(*args2, **kwargs)

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

try:
    print_es
except NameError:
    def print_es(*args, **kwargs):
        import re as _re
        print(*[_re.sub(r"(\d)\.(\d)", r"\1,\2", a) if isinstance(a, str) else a for a in args], **kwargs)
# Signo menos tipográfico (−) en todo texto impreso: «-5» → «−5» (no toca guiones entre palabras, rangos ni 1e-3)
import builtins as _bi_br
if not getattr(_bi_br.print, '_menos_br', False):
    def _print_br(*a, _orig=_bi_br.print, **k):
        import re as _re
        _orig(*[_re.sub(r'(?<![\w.,\-])-(?=\d)', '−', x) if isinstance(x, str) else x for x in a], **k)
    _print_br._menos_br = True
    _bi_br.print = _print_br
# --- Cuestionarios Física BR v3: opciones barajadas, crédito por intento, nota sobre el total ---
import random as _rnd_br
try:
    PESOS_BR
except NameError:
    PESOS_BR = {}      # id_preg -> peso de cada pregunta mostrada
try:
    INTENTOS_BR
except NameError:
    INTENTOS_BR = {}   # id_preg -> [intentos, resuelta]
_CREDITO_BR = {1: 1.0, 2: 0.5, 3: 0.25}   # crédito según el intento en que se acierta
_MAX_INT_BR = 3

def _evaluar_intento(id_preg, ok, peso, explicacion, msg_ok, msg_mal):
    est = INTENTOS_BR.setdefault(id_preg, [0, False])
    if est[1]:
        print("✔ Ya resolviste esta pregunta; tu crédito quedó registrado.")
        if explicacion: print_es("→ " + explicacion)
        return
    if est[0] >= _MAX_INT_BR:
        print("⛔ Agotaste los 3 intentos con crédito: esta pregunta ya no suma.")
        if explicacion: print_es("→ " + explicacion)
        return
    est[0] += 1
    n = est[0]
    if ok:
        est[1] = True
        cred = _CREDITO_BR.get(n, 0.0)
        REGISTRO_BR[id_preg] = (cred, float(peso))
        print_es(msg_ok + f"  (intento {n}: crédito {int(cred*100)} %)")
        if explicacion: print_es("→ " + explicacion)
    else:
        REGISTRO_BR.setdefault(id_preg, (0.0, float(peso)))
        if n < _MAX_INT_BR:
            print(msg_mal + f"  Intento {n} de {_MAX_INT_BR}. Revisa tu razonamiento antes de volver a intentar.")
        else:
            print(msg_mal + " Agotaste los 3 intentos: esta pregunta ya no suma. Estudia la explicación:")
            if explicacion: print_es("→ " + explicacion)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    orden = list(range(len(opciones))); _rnd_br.shuffle(orden)   # orden distinto en cada sesión
    correcta = orden.index(indice_correcto)
    letras = "abcdefgh"
    etiquetas = [f"{letras[i]}) {opciones[j]}" for i, j in enumerate(orden)]
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=etiquetas, value=None, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            if radio.index is None:
                print("⚠ Elige una opción."); return
            _evaluar_intento(id_preg, radio.index == correcta, peso, explicacion,
                             "✅ ¡Correcto!", "❌ Incorrecto.")
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.strip().replace(",", "."))
            except ValueError: print("⚠ Escribe un número (usa coma o punto decimal)."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            _evaluar_intento(id_preg, ok, peso, explicacion,
                             f"✅ ¡Correcto!  ({valor_correcto:.3g} {unidad})", "❌ Fuera de rango.")
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not PESOS_BR:
                print("No se encontraron cuestionarios en este taller."); return
            peso_total = sum(PESOS_BR.values())
            peso_ok = sum(float(c)*p for c, p in REGISTRO_BR.values())
            nota = 5.0*peso_ok/peso_total if peso_total > 0 else 0.0
            total = len(PESOS_BR)
            resp = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0])[0] > 0)
            prim = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0, False])[1] and INTENTOS_BR[k][0] == 1)
            print("="*46)
            print("          CALIFICACIÓN FORMATIVA")
            print("="*46)
            print(f"  Preguntas respondidas:       {resp} de {total}")
            print(f"  Acertadas al primer intento: {prim}")
            print_es(f"  Puntaje ponderado:           {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*46)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*46)
            if resp < total: print(f"  ⚠ Faltan {total-resp} preguntas: cuentan como 0.")
            if nota >= 3.5: print("  ✅ ¡Buen trabajo! Dominas los conceptos.")
            elif nota >= 3.0: print("  🟡 Vas bien, repasa lo que fallaste.")
            else: print("  🔴 Repasa la teoría y vuelve a los puntos.")
            print("\n  Crédito por pregunta: 100 % al primer intento,")
            print("  50 % al segundo, 25 % al tercero. Predice y")
            print("  razona ANTES de verificar: adivinar cuesta nota.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>📊 Tu calificación del taller</h3>"
        "<i>La nota se calcula sobre TODAS las preguntas del taller y premia acertar al primer intento.</i>"),
        boton, salida]))


# Calificación formativa del taller
calificacion_final(total_preguntas=16)
